# song-lingo 연습 화면 (practice)

시연 화면입니다. 위에서 아래로 실행합니다. 구현된 모듈이 늘어날 때마다 셀을 추가합니다.

- 녹음은 노트북에서 직접 하지 않고 **미리 녹음한 파일 경로**를 지정합니다.
- 이 노트북의 출력에는 목소리·경로·key가 남을 수 있으므로 **커밋 전에 출력을 지우고** `scripts/scan_outputs.py`로 확인합니다.

## 큰 흐름 지도: 이 노트북은 "화면"이고, 일은 `.py` 모듈이 합니다

이 노트북의 셀은 **모듈의 함수를 부르고 결과를 보여 주기만** 합니다. 소리 읽기·인식·채점·그림 계산은 프로젝트 루트의 `.py` 파일이 합니다. 각 코드 셀 맨 위의 `# ──` 주석 블록에 **그 셀이 하는 일, 부르는 모듈·함수, STT 모델을 쓰는지**를 적어 두었습니다.

### 데이터가 흐르는 순서

```text
sentences.csv ─ tts.py ─(캐시 data/tts_cache/sNN.wav; 없을 때만 OpenAI TTS API 호출 = 과금)─▶ 기준 음성

내 녹음 data/recordings/sNN_takeK.wav   (녹음은 scripts/record_takes.py + recorder.py 가 만듭니다. 노트북에서는 녹음하지 않음)
   ├─ audio.py(읽기 → 모노 → 16 kHz) ─ quality.py ─────────▶ 녹음 상태 점검 (경고는 안내일 뿐, 계속 진행)
   └─ stt.py ─ faster-whisper 모델(로컬 CPU) ─────────────▶ 인식 문장
                  └─ scoring.py (정답 문장과 비교) ────────▶ WER/CER, 치환·삭제·삽입(S·D·I), 틀린 부분 [정답→인식]

틀린 부분 + 단어별 시간(stt.recognize_words) ─ align.py ─▶ TTS·내 녹음의 시간 구간
practice.py (위 순서를 이어 부름, 결과를 dict로 돌려줌) ─▶ report.py(글) / visualisation.py(그림) ─▶ 이 노트북의 출력
```

### 모듈이 맡은 일과 노트북에서 부르는 함수

| 모듈 | 책임 (한 줄) | 노트북에서 부르는 것 |
|---|---|---|
| `tts.py` | 문장 목록 읽기, 기준 음성(TTS) 만들기·캐시 | `load_sentences`, `tts_wav` |
| `stt.py` | **STT 모델을 읽고(`load_model`) 녹음을 인식(`recognize`, `recognize_words`)**, 녹음 파일 찾기 | `load_model`, `recognize`, `find_recording`, `available_models` |
| `scoring.py` | 정규화, WER/CER, 틀린 부분 표시 (글자만 다룸, 소리·API 없음) | `wer`, `cer`, `format_diff` |
| `audio.py` | 어떤 녹음이든 16 kHz 모노 파형으로 | `load_audio` |
| `quality.py` | 녹음 상태 점검(음량·무음·클리핑·말한 시간) | `check_file`, `format_report` |
| `align.py` | 틀린 단어를 소리의 시간 구간으로 | (`practice.py`가 부름) |
| `practice.py` | **연습 한 번의 순서를 이어 붙임** (계산은 하지 않음) | `prepare`, `read`, `dictation`, `locate_errors` (노트북에서는 `flow`라고 부름) |
| `report.py` | 결과 dict → 읽을 글 | `header`, `tts_status`, `result`, `error_blocks` |
| `visualisation.py` | 스펙트로그램·Log-Mel 계산과 그림 | `logmel`, `plot_sound_pair`, `plot_error_spans` |
| `model_compare.py` | 같은 녹음을 여러 모델 크기로 인식·채점해 비교 | `compare_models` |

### "STT 모델은 정확히 어디서 어떤 것을 불러옵니까?"

- **불러오는 코드는 한 곳**입니다: `stt.py`의 `load_model(size)` 안의 `WhisperModel(size, device="cpu", compute_type="int8", download_root="models")`. 노트북 셀은 이 함수를 직접 부르거나(`stt.load_model()`), 인식 함수(`recognize`)가 대신 부릅니다.
- **어떤 크기인가**: `stt.py`의 `MODEL_SIZE = "tiny"`가 기본입니다. 크기를 바꾸려면 `recognize(..., model_size="small")`처럼 호출 때 넘기거나(실험2·3), 기본값 자체를 바꾸려면 `stt.py`의 이 한 줄을 고칩니다.
- **한 번 읽으면 다시 읽지 않습니다**: 읽은 모델은 `stt._models`에 크기별로 보관됩니다. 파일은 `models/`(git 제외)에 있고, 없는 크기는 처음 쓸 때 인터넷으로 내려받습니다(실험2의 `DOWNLOAD_MORE = True`가 `base`·`small`을 받는 곳).
- **과금**: STT는 **로컬 CPU 실행이라 과금이 없습니다.** OpenAI API(과금)는 **TTS(기준 음성 만들기)에서만**, 캐시에 없는 문장일 때만 호출됩니다.

| 노트북 위치 | 부르는 것 | 쓰는 STT 모델 | 인식 옵션 |
|---|---|---|---|
| S0-3 모델 로딩 확인 | `stt.load_model()` | `tiny` | 읽기만 하고 인식은 안 함 |
| S2 내 녹음 인식 | `stt.recognize(path)` | `tiny` | VAD 끔 |
| 대조 실험1 (TTS를 인식) | `recognize(TTS wav)` | `tiny` | VAD 끔 |
| 틀린 구간 표시 | `flow.locate_errors` → `stt.recognize_words` (내 녹음, TTS 두 번) | `tiny` | **VAD 켬**, 단어별 시간 |
| 실험2 모델 크기 비교 | `model_compare.compare_models(SIZES)` → `stt.load_model(size)` + `stt.recognize(model_size=size)` | `tiny`·`base`·`small` | VAD 끔 |
| 실험3 녹음 30건 | `compare_models([기본 모델])` | `tiny` | VAD 끔 |
| S4 `run_practice(…, "read")` | `flow.read` → `stt.recognize` | `tiny` | VAD 끔 |
| S4 `run_practice(…, "dictation")` | **STT를 쓰지 않음** (내가 적은 글을 바로 채점) | 없음 | |

### 알아 둘 규칙 두 가지

1. **위에서 아래로 실행합니다.** 뒤쪽 셀은 앞 셀이 만든 변수를 씁니다(예: `sentences`는 S1, `hyps`는 S2, `flow`·`report`는 틀린 구간 표시 셀에서 만들어짐). 각 셀 주석의 "가져오는 것"에 적어 두었습니다.
2. **`.py`나 `.env`를 고친 뒤에는 커널을 재시작합니다.** 커널은 한 번 import한 모듈을 다시 읽지 않습니다(예: `scoring.py`에 인자를 추가했는데 커널이 옛날 버전을 들고 있어 `unexpected keyword argument` 오류가 났음).

In [ ]:
# ── S0-0. 환경 준비 ───────────────────────────────────────────────────
# 역할       : 프로젝트 루트로 이동 → 어느 파이썬으로 실행 중인지 확인 → .env의 API 키를 환경 변수로 읽기 → 채점 모듈 불러오기
# 담당 모듈    : scoring.py (여기서는 import만 합니다)
# STT 모델   : 사용 안 함 / API 호출·과금 없음
# 이 노트북은 experiments/ 안에 있고 모듈(scoring.py 등)과 data/ 폴더는 프로젝트 루트에 있습니다.
# 그래서 위 폴더로 올라가며 루트(practice.py가 있는 곳)를 찾아 그곳으로 이동하고 import 경로에 넣습니다.
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "practice.py").exists() and root != root.parent:
    root = root.parent                      # 위 폴더로 올라가며 프로젝트 루트를 찾는다
os.chdir(root)                              # 이후 셀의 상대 경로(data/recordings 등, sentences.csv)가 루트 기준이 된다
sys.path.insert(0, str(root))               # `import scoring` 같은 프로젝트 모듈을 찾게 한다
print("프로젝트 루트:", root.name)           # 폴더 이름만 출력(전체 경로에는 사용자 이름이 들어감)
print(sys.executable)   # .../envs/voice-ai/bin/python 이어야 합니다

# .env 파일을 읽는 도구(python-dotenv). 파일 안의 OPENAI_API_KEY를 환경 변수로 넣어 줍니다.
from dotenv import load_dotenv
# 이 키는 tts.py가 OpenAI TTS를 부를 때만 씁니다. STT(Whisper)는 내 컴퓨터에서 도는 모델이라 키가 필요 없습니다.
load_dotenv()            # .env 의 OPENAI_API_KEY 를 읽습니다 (값은 출력하지 않습니다)

# scoring.py: 정답 문장과 인식 결과를 비교해 WER/CER을 계산합니다. 아래 모든 채점 셀이 이 모듈을 씁니다.
# 주의: 커널은 import한 .py를 다시 읽지 않습니다. scoring.py 같은 파일을 고쳤으면 커널을 재시작(Restart)해야 반영됩니다.
import scoring

## S0. 준비 확인

1. 채점 동작 확인 (API 호출 없음)

In [ ]:
# ── S0-1. 채점이 동작하는지 확인 (API 호출 없음) ────────────────────────────────
# 역할       : 정답 ref와 인식 결과 hyp(여기서는 내가 일부러 틀리게 쓴 글)를 비교해 오류율과 틀린 부분을 봅니다.
# 담당 모듈    : scoring.py — wer(): 정규화 → 단어로 나눔 → 편집 거리 → 치환(S)·삭제(D)·삽입(I) 개수와 오류율
# 입력→출력    : 글자 두 개 → 오류율, S·D·I, 틀린 부분 표시 (소리·STT·API가 전혀 없는 순수 계산)
# STT 모델   : 사용 안 함
ref = "The cat sat on the mat."
hyp = "the cat sit on mat"   # 일부러 틀리게 쓴 인식 결과: sat→sit(치환 1), the 빠짐(삭제 1)

# wer(): 소문자·구두점 제거(+ 축약형·숫자 표기 통일) 후 단어 단위로 비교합니다. 돌려주는 r은 dict:
# {"S", "D", "I", "N"(정답 단어 수), "rate"(=(S+D+I)/N), "ops"(단어별 일치/치환/삭제/삽입 목록)}
r = scoring.wer(ref, hyp)
print(f"WER {r['rate']:.3f}  (치환 {r['S']}, 삭제 {r['D']}, 삽입 {r['I']} / 정답 {r['N']}단어)")
# format_diff(): ops를 [정답→인식](치환) / [-정답-](삭제) / [+인식+](삽입) 표시 한 줄로 바꿔 줍니다.
print(scoring.format_diff(r['ops']))

2. 키가 읽히는지(값은 출력하지 않음), 

In [ ]:
# ── S0-2. API 키가 읽혔는지 확인 ──────────────────────────────────────────
# 역할       : .env의 OPENAI_API_KEY가 환경 변수에 들어왔는지만 확인합니다(값은 출력하지 않음).
# 왜 필요한가   : 키는 TTS(기준 음성 만들기)에서만 필요합니다. 캐시만 쓰는 연습에서는 없어도 동작합니다.
# STT 모델   : 사용 안 함 / API 호출 없음
import os
print("키 존재:", bool(os.getenv("OPENAI_API_KEY")))   # bool()로 True/False만 출력해서 키 값이 화면·저장된 출력에 남지 않습니다
print("커널 python:", sys.executable)

In [ ]:
# ── (진단용) 어떤 OPENAI_API_KEY가 적용되는지 확인 ─────────────────────────────
# 용도       : 평소에는 주석 처리해 둡니다. .env를 고쳤는데 같은 401 오류가 날 때 주석을 풀어 실행합니다.
# 대신       : 터미널에서: python scripts/check_api_key.py  (값은 앞 7자·끝 4자만 보여 줌)
# import os
# from dotenv import dotenv_values, load_dotenv
# def mask(v): return "(없음)" if not v else f"{v[:7]}…{v[-4:]} (길이 {len(v)})"

# before = os.environ.get("OPENAI_API_KEY")      # 커널이 이미 들고 있는 값(load_dotenv 전)
# load_dotenv()
# print("커널의 실제 값 :", mask(os.environ.get("OPENAI_API_KEY")))
# print(".env 파일의 값 :", mask(dotenv_values(".env").get("OPENAI_API_KEY")))
# print("load_dotenv 전 :", mask(before))

3. Whisper `tiny` 모델이 로딩되는지 봅니다. 모델은 학습 저장소의 `models/`를 복사해 재사용합니다.

In [ ]:
# ── S0-3. Whisper(STT) 모델이 읽히는지 확인 ────────────────────────────────
# 역할       : STT 모델을 미리 한 번 읽어 두고, 읽는 데 걸리는 시간을 잽니다. 이 프로젝트에서 STT 모델 객체를 만드는 곳은
#            stt.load_model() 단 한 곳입니다(여기서 부르든, 인식 함수가 대신 부르든 같은 함수가 같은 일을 합니다).
# 호출 경로    : 이 셀 → stt.load_model() → faster_whisper.WhisperModel("tiny", device="cpu", compute_type="int8", download_root="models")
# 어떤 모델    : stt.MODEL_SIZE = "tiny" (기본). 파일은 models/ 폴더(git 제외)에 있고, 없는 크기는 처음 쓸 때만 내려받습니다.
# 이후       : 읽은 모델은 stt._models에 크기별로 보관되어 다시 읽지 않습니다. 아래 recognize()가 이 모델을 그대로 씁니다.
# 과금       : 없음 (내 컴퓨터 CPU에서 실행, OpenAI STT는 쓰지 않음)
import time
import stt   # stt.py: 녹음 찾기와 faster-whisper 인식

t0 = time.time()
model = stt.load_model()   # tiny + int8, 처음 한 번만 읽어 stt 모듈에 보관합니다   # 인자를 안 주면 기본 크기 tiny. stt.load_model("small")처럼 크기를 넘기면 그 크기를 읽습니다(실험2·3)
print(f"모델 로딩 성공 ({time.time() - t0:.1f}초)")

## S1. 연습 문장과 기준 음성(TTS) 준비

### 1개 생성 테스트

`sentences.csv`에서 문장을 읽고, 1번 문장의 기준 음성을 만듭니다. 모델과 voice는 013 노트북 설정(`gpt-4o-mini-tts`, `alloy`)을 따랐고, 영어가 되는지는 이 셀에서 처음 확인합니다. **실제 과금되므로** 결과를 `data/tts_cache/`에 저장해 두고 있으면 다시 호출하지 않습니다.

In [ ]:
# ── S1-1. 문장 목록 읽기 + 기준 음성(TTS) 1개 확인 ─────────────────────────────
# 역할       : sentences.csv의 10문장을 읽고, 1번 문장의 기준 음성(TTS)을 준비해 재생 버튼을 보여 줍니다.
# 호출 경로    : tts.load_sentences() → sentences.csv를 dict 목록으로
#            tts.tts_wav(문장, 경로) → 파일이 있으면 경로만 돌려주고(과금 없음), 없을 때만 OpenAI TTS API를 부릅니다(과금)
# TTS 설정   : gpt-4o-mini-tts, voice alloy (tts.py의 TTS_MODEL·TTS_VOICE). 만든 음성은 data/tts_cache/sNN.wav에 저장
# STT 모델   : 사용 안 함
# 만드는 변수   : sentences(이후 거의 모든 셀이 씀), tts_wav, CACHE_DIR
from pathlib import Path
from IPython.display import Audio, display

import tts   # tts.py: 문장 읽기, TTS 호출과 캐시, 호출 횟수(tts.calls)
from tts import tts_wav, CACHE_DIR

# sentences: [{"id", "lang", "sentence", "meaning_ko", "probe", ...}, ...] 10개
sentences = tts.load_sentences()
for s in sentences:
    print(s["id"], s["sentence"])

# 첫 번째 문장(id 1) 하나만 골라 기준 음성을 확인합니다
s1 = sentences[0]
# 캐시 파일 이름 규칙: id 1 → s01.wav (두 자리). 있으면 API 호출 0회
ref_path = tts_wav(s1["sentence"], CACHE_DIR / f"s{int(s1['id']):02d}.wav")
print("기준 음성:", ref_path, f"({ref_path.stat().st_size // 1024} KB), API 호출 {tts.calls}회")
# Audio: 이 노트북 안에서 소리를 재생하는 버튼. 주의: 출력에 소리 데이터가 박히므로 커밋 전에 출력을 지웁니다
display(Audio(filename=str(ref_path)))

### 연습 문장 10개의 기준 음성(TTS)

`sentences.csv`의 10문장 중 `data/tts_cache/`에 기준 음성이 없는 문장을 찾고, 필요하면 만듭니다. 문장을 고른 기준과 문장별 의도(`probe`)는 `experiments/WER, CER/연습_문장_데이터_설명.md`에 있습니다.

- 아래 첫 셀은 **확인만** 합니다(API 호출·과금 없음).
- 두 번째 셀은 **실제로 과금**됩니다(문장 1개당 API 1회). 그래서 `GENERATE_MISSING_TTS = False`가 기본이고, `True`로 바꿔 실행하면 **캐시에 없는 문장만** TTS API로 만듭니다. 끝나면 다시 `False`로 돌려 둡니다(`scripts/check_notebook.py`로 노트북을 점검할 때 과금되지 않게 하려는 것입니다).
- 오디오 플레이어는 출력에 소리 데이터(base64)가 박히므로 **커밋 전에 출력을 지우고** `scripts/scan_outputs.py`로 확인합니다.

In [ ]:
# ── S1-2. 기준 음성이 없는 문장 찾기 (확인만, 과금 없음) ────────────────────────────
# 역할       : 10문장 중 data/tts_cache/에 음성 파일이 없는 문장을 찾아 보여 줍니다.
# 담당 모듈    : tts.py (load_sentences, CACHE_DIR)
# STT 모델   : 사용 안 함 / API 호출 없음
# 만드는 변수   : missing_tts → 바로 아래 생성 셀이 이 목록을 씁니다
import tts

all_sentences = tts.load_sentences()
# 파일 존재 여부(exists)만 검사합니다. 목록이 비어 있으면 10문장의 음성이 모두 캐시에 있다는 뜻
missing_tts = [s for s in all_sentences if not (tts.CACHE_DIR / f"s{int(s['id']):02d}.wav").exists()]
print(f"문장 {len(all_sentences)}개 중 기준 음성이 없는 문장 {len(missing_tts)}개 (만들면 API 호출 {len(missing_tts)}회):")
for s in missing_tts:
    print(f"  {s['id']:>2}  {s['sentence']}   ← {s.get('probe', '')}")

In [ ]:
# ── S1-3. 없는 문장의 기준 음성 만들기 (여기서만 TTS 과금) ──────────────────────────
# 역할       : S1-2에서 찾은 문장(missing_tts)에 대해서만 OpenAI TTS API를 불러 음성을 만들고 캐시에 저장합니다.
# 호출 경로    : 이 셀 → tts.tts_wav(문장, 캐시 경로) → OpenAI TTS API (gpt-4o-mini-tts, alloy) → data/tts_cache/sNN.wav
# 과금       : 문장 1개당 API 1회. missing_tts가 비어 있으면 아무것도 만들지 않으므로 과금 0회
# STT 모델   : 사용 안 함
# 스위치: True면 만들고 False면 건너뜁니다. 지금은 True이지만 10문장 음성이 이미 모두 있어 아무 일도 일어나지 않습니다.
# 새 문장을 sentences.csv에 추가했을 때만 실제 호출(과금)이 생기므로, 평소에는 False로 두는 것이 안전합니다.
GENERATE_MISSING_TTS = True   # True로 바꿔 실행하면 위에서 찾은 문장만 TTS API로 만듭니다(과금). 끝나면 False로 되돌립니다.

if GENERATE_MISSING_TTS:
    from IPython.display import Audio, display
    print(f"기준 음성이 없는 문장 {len(missing_tts)}개")
    for s in missing_tts:
        # tts_wav: 파일이 있으면 경로만 돌려주고, 없을 때만 API를 부릅니다(부르면 tts.calls가 1 늘어남)
        path = tts.tts_wav(s["sentence"], tts.CACHE_DIR / f"s{int(s['id']):02d}.wav")   # 이미 있으면 API를 부르지 않음
        print(f"{s['id']:>2} {s['sentence']}  ({path.stat().st_size // 1024} KB)")
        display(Audio(filename=str(path)))
    print("이 커널의 API 호출 횟수:", tts.calls)
else:
    print("GENERATE_MISSING_TTS = False라서 만들지 않았습니다 (과금 없음).")

**참고: 위 확인 셀과 생성 셀의 출력이 서로 달라 보일 때**

- `tts.calls`는 **커널을 시작한 뒤 실제로 API를 부른 횟수의 누적값**입니다. 셀마다 0으로 돌아가지 않으므로, 생성이 끝난 뒤에는 어느 셀에서 찍어도 같은 값(이 프로젝트에서는 8회: 2번과 4~10번)이 나옵니다. "이번 셀에서 부른 횟수"가 아닙니다. 만들 문장이 없어 반복문이 한 번도 돌지 않았어도 이 값은 그대로입니다.
- `missing_tts`는 **확인 셀을 실행한 시점의 목록**(스냅샷)입니다. 생성 셀은 이 목록을 그대로 쓰므로, 확인 셀을 다시 실행하지 않으면 오래된 목록을 봅니다. 파일이 이미 있으면 `tts_wav`가 API를 부르지 않아 과금은 늘지 않지만 출력이 헷갈릴 수 있습니다. 그래서 **생성 셀을 돌리기 전에 확인 셀을 먼저 다시 실행**합니다.
- 각 셀의 출력은 **그 셀을 마지막으로 실행한 시점**의 값입니다. 셀 순서를 바꾸거나 일부만 다시 실행하면 위아래 출력이 서로 다른 시점의 값이 됩니다. 예: 확인 셀은 생성이 끝난 뒤에 다시 돌려 "0개", 생성 셀은 먼저 돌렸던 실행의 누적 카운터 "8".
- 10문장의 기준 음성은 2026-11-02 22:19에 8개를 만들어 모두 `data/tts_cache/`에 있습니다(파일 수정 시각으로 확인). 끝났으면 `GENERATE_MISSING_TTS`를 `False`로 되돌립니다. 그래야 `scripts/check_notebook.py`가 과금 없이 통과합니다.
- **이번 실행에서 부른 횟수만** 보려면 생성 전에 `before = tts.calls`를 저장하고, 끝난 뒤 `tts.calls - before`를 출력합니다.

## S2. 내 녹음 인식(faster-whisper)

`data/recordings/`의 `sNN_take1.*`(wav·m4a·mp3 모두 가능)를 읽어 Whisper `tiny`로 인식합니다(`language="en"`, `beam_size=1`). 노트북에서 직접 녹음하지 않고 미리 녹음한 파일을 씁니다. 이 셀의 출력에는 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.

In [ ]:
# ── S2. 내 녹음 인식 (STT) — 문장 1~10의 take1 ────────────────────────────
# 역할       : 미리 녹음한 data/recordings/sNN_take1.* 를 읽어 Whisper로 문장으로 바꿉니다.
# 호출 경로    : stt.find_recording(문장 id) → 녹음 파일 경로 찾기(확장자 무관)
#            stt.recognize(path) → load_model(None)=tiny → model.transcribe(path, language="en", beam_size=1, vad_filter=False) → (인식 문장, 걸린 초)
# STT 모델   : tiny (stt.MODEL_SIZE 기본값), CPU·int8·beam_size=1, VAD 끔, 영어 고정. S0-3에서 읽어 둔 모델을 그대로 씁니다.
# 과금       : 없음 (로컬 실행)
# 가져오는 것   : sentences (S1-1)
# 만드는 변수   : hyps = {문장 id: 인식 문장} → S3 채점, 대조 실험1이 씁니다
# 참고       : take2·take3까지 포함한 30건 전체는 아래 '실험3'에서 한꺼번에 인식합니다.
# REC_DIR(녹음 폴더), find_recording(녹음 찾기), recognize(인식) — 모두 stt.py의 함수입니다
from stt import REC_DIR, find_recording, recognize   # stt.py로 옮긴 함수들

# 인식 결과를 모아 둘 dict. 키가 문장 id, 값이 인식된 문장
hyps = {}   # {문장 id: 인식 결과}
for s in sentences:
    # 녹음이 없으면 None → 아래에서 '녹음 없음'을 출력하고 건너뜁니다
    path = find_recording(s["id"])
    if path is None:
        print(f"[{s['id']}] 녹음 없음: {REC_DIR}/s{int(s['id']):02d}_take1.*")
        continue
    # 실제로 STT가 돌아가는 줄입니다(수 초). 돌려받는 sec은 인식에 걸린 시간
    hyps[s["id"]], sec = recognize(path)
    print(f"[{s['id']}] 정답: {s['sentence']}")
    print(f"    인식: {hyps[s['id']]}   ({path.suffix}, {sec:.1f}초)")

## S3. 내 녹음 채점 + 틀린 부분 표시

정답 문장과 인식 결과를 `scoring`으로 비교합니다. 점수는 발음 점수가 아니라 **문장 인식 일치도**의 오류율입니다. 오류율은 1을 넘을 수 있어 "일치율"로 바꿔 표시하지 않고 치환(S)·삭제(D)·삽입(I) 개수를 함께 보여 줍니다. 틀린 부분은 `[정답→인식]`(치환), `[-정답-]`(삭제), `[+인식+]`(삽입)으로 표시합니다.

In [ ]:
# ── S3. 내 녹음 채점 + 틀린 부분 표시 ────────────────────────────────────────
# 역할       : 정답 문장과 S2의 인식 결과(hyps)를 비교해 오류율과 틀린 부분을 출력합니다.
# 호출 경로    : scoring.wer() — 단어 단위 / scoring.cer() — 글자 단위 / scoring.format_diff() — 틀린 부분 표시
# STT 모델   : 사용 안 함 (S2의 결과를 비교만 합니다)
# 가져오는 것   : sentences (S1-1), hyps (S2), scoring (S0-0)
# 읽는 법     : 점수는 발음 점수가 아니라 '문장 인식 일치도'의 오류율입니다. S·D·I는 치환·삭제·삽입 개수입니다.
for s in sentences:
    if s["id"] not in hyps:
        continue
    # w, c는 dict: {"S", "D", "I", "N", "rate", "ops"}. rate = (S+D+I)/N (1을 넘을 수 있음)
    w = scoring.wer(s["sentence"], hyps[s["id"]])
    c = scoring.cer(s["sentence"], hyps[s["id"]])
    print(f"문장 #{s['id']}: \"{s['sentence']}\"")
    print(f"Whisper(tiny) 인식: \"{hyps[s['id']]}\"")
    print(f"WER {w['rate']:.3f}  (치환 {w['S']}, 삭제 {w['D']}, 삽입 {w['I']} / 정답 {w['N']}단어)")
    print(f"CER {c['rate']:.3f}  (치환 {c['S']}, 삭제 {c['D']}, 삽입 {c['I']} / 정답 {c['N']}글자)")
    # [정답→인식] 치환 / [-정답-] 삭제 / [+인식+] 삽입
    print(scoring.format_diff(w["ops"]))
    print()

### 대조 실험1) TTS 기준 음성을 같은 Whisper에 넣어 보기

3번 문장의 `we'll sing` 구간이 두 번 모두 틀렸습니다. 원인이 내 녹음·발음인지 `tiny` 모델의 한계인지 가르기 위해, TTS로 만든 3번 문장 기준 음성을 같은 설정으로 인식시킵니다. 표본이 1개(문장 1개, voice 1개, 1회)라서 **일반화하지 않고 이 문장에서의 참고 결과로만** 봅니다.

In [ ]:
# ── 대조 실험1. TTS 기준 음성을 같은 Whisper에 넣어 보기 ──────────────────────────
# 목적       : 3번 문장을 내 녹음은 틀리게, TTS 음성은 맞게 인식하는지 비교해 원인이 '내 녹음'인지 '모델 한계'인지 가립니다.
# 호출 경로    : tts_wav(…) → 캐시의 TTS wav 경로(API 0회) → recognize(p3) → 같은 tiny 모델·같은 옵션으로 인식 → scoring.wer
# STT 모델   : tiny, VAD 끔 (S2와 완전히 같은 설정이어야 비교가 공정합니다)
# 가져오는 것   : sentences (S1-1), tts_wav·CACHE_DIR (S1-1), recognize (S2), hyps (S2), scoring (S0-0)
s3 = sentences[2]
p3 = tts_wav(s3["sentence"], CACHE_DIR / f"s{int(s3['id']):02d}.wav")   # 이미 있으면 API를 부르지 않음
# TTS 음성을 STT에 넣는 줄 (기준 음성도 Whisper가 맞게 알아듣는지 확인)
tts_hyp, sec = recognize(p3)
w = scoring.wer(s3["sentence"], tts_hyp)

print(f"정답          : {s3['sentence']}")
print(f"TTS 인식      : {tts_hyp}   ({sec:.1f}초, 이 커널의 API 호출 {tts.calls}회)")
print(f"내 녹음 인식  : {hyps[s3['id']]}")
print(f"TTS  WER {w['rate']:.3f} (치환 {w['S']}, 삭제 {w['D']}, 삽입 {w['I']} / {w['N']}단어)")
print(scoring.format_diff(w["ops"]))

#### 같은 문장의 TTS와 내 녹음을 소리 그림으로 비교

3번 문장(`We'll sing together until the morning comes.`)의 TTS 기준 음성과 내 녹음을 **같은 설정**으로 그립니다. 수업 코드를 가져와 썼습니다.

| 수업 내용 | 출처 | 이 그림에서의 역할 |
|---|---|---|
| WAV 읽기, 샘플링 레이트, 모노·16 kHz 변환 | 02, lesson04 4-D | 48 kHz(내 녹음)와 24 kHz(TTS)를 같은 16 kHz 모노로 맞춤 |
| STFT (창 512, 이동 128, hann) | 03 `compute_stft`, lesson04 | 시간별 주파수 분포(스펙트로그램) |
| Mel 필터 40개 + 로그 | lesson04 `logmel` | Log-Mel (Whisper가 모델 입력으로 쓰는 특징의 교육용 버전) |

**읽는 법**: 이 그림은 **참고 정보이며 평가가 아닙니다.** 스펙트로그램이 다르다고 해서 발음이 틀렸다는 뜻이 아닙니다. 두 소리의 길이·말 속도·무음 구간·에너지가 모이는 위치가 어떻게 다른지 눈으로 보는 용도입니다. 비교를 위해 두 파형은 최대 진폭을 같게 맞추고, 같은 색 범위로 그립니다.

In [ ]:
# ── 소리 그림 준비 (import만) ────────────────────────────────────────────
# 역할       : 그림에 쓰는 도구를 불러오고, 프로젝트 공통 그림 스타일(초록 계열)을 적용합니다.
# 담당 모듈    : theme.py(색·글꼴), audio.py(load_audio), visualisation.py(logmel, plot_sound_pair, plot_error_spans)
# STT 모델   : 사용 안 함
import numpy as np
import matplotlib.pyplot as plt

# theme.py: 그래프 공통 색·스타일
import theme
# 프로젝트 공통 그림 스타일을 적용합니다(이후 그리는 모든 그림에 적용)
theme.apply_style()   # 프로젝트 공통 색·글꼴 (theme.py, 초록 계열)

# 수업 코드를 모듈로 옮겼습니다(동작은 같음).
# audio.load_audio: 어떤 녹음 파일이든 16 kHz 모노 파형으로 바꿉니다(수업 lesson04 4-D의 변환 순서)
from audio import load_audio                  # audio.py: 어떤 녹음 파일이든 16 kHz 모노 float32 파형으로 (lesson04 4-D 순서)
# visualisation.py: logmel(STFT·Mel 계산), plot_sound_pair(두 소리 그림), plot_error_spans(틀린 구간 표시 그림)
from visualisation import logmel, plot_sound_pair, plot_error_spans
# visualisation.py: STFT 파워 → Mel 필터 → Log-Mel(dB) 계산(lesson04 실습 2-F·3-B)과 그림 그리기

In [ ]:
# ── 소리 그림 비교. 같은 문장의 TTS와 내 녹음 ────────────────────────────────────
# 역할       : 3번 문장의 TTS 기준 음성과 내 녹음을 같은 설정으로 파형·스펙트로그램·Log-Mel로 그립니다(참고 정보, 평가 아님).
# 호출 경로    : audio.load_audio(경로) → 16 kHz 모노 파형 / visualisation.logmel(파형, 16000) → STFT·Log-Mel 숫자 / visualisation.plot_sound_pair(…) → 그림
# STT 모델   : 사용 안 함 (소리 신호 처리만)
# 가져오는 것   : sentences·CACHE_DIR (S1-1), find_recording (S2), np·plt·logmel·plot_sound_pair (바로 위 준비 셀)
sid = sentences[2]["id"]
# 비교할 두 소리: (이름, 파일 경로)
sources = [("TTS 기준 음성", CACHE_DIR / f"s{int(sid):02d}.wav"), ("내 녹음", find_recording(sid))]

sounds = []
for name, p in sources:
    # x: 16 kHz 모노 float 파형, sr0·ch: 원본 표본화 주파수·채널 수
    x, sr0, ch = load_audio(p)
    # 최대 진폭을 같게 맞춰 소리 크기 차이는 지우고 모양만 비교합니다
    x = x / max(np.abs(x).max(), 1e-9) * 0.9          # 최대 진폭을 맞춰 모양만 비교
    f = logmel(x, 16000)
    sounds.append((name, x))
    print(f"{name}: 원본 {sr0} Hz / {ch}채널 → 16000 Hz 모노, {len(x) / 16000:.2f}초, "
          f"STFT {f['power'].shape}, Log-Mel {f['log_mel'].shape}  (주파수·Mel칸, 시간)")

# 그리기는 visualisation.plot_sound_pair가 합니다(두 소리를 같은 설정·같은 색 범위로).
# 그리기는 visualisation.py가 합니다. 두 소리를 같은 설정·같은 색 범위로 그립니다
plot_sound_pair(sounds, f'문장 #{sid}: "{sentences[2]["sentence"]}"  (참고 정보, 평가 아님; STFT 창 512·이동 128, Mel 40, 같은 색 범위)')
plt.show()

**소리 그림 결과 검토**

표시: **[확인]** 그림에서 바로 보이는 것, **[추정]** 해석, **[미확인]** 이 그림만으로는 알 수 없는 것. 이 그림은 참고 정보이며 평가가 아닙니다.

| 관찰 | 근거 | 해석 |
|---|---|---|
| 말하는 구간의 길이는 비슷합니다. TTS는 약 0.25~3.2초(약 3초), 내 녹음은 약 1.3~4.5초(약 3.2초)입니다. | **[확인]** 파형·스펙트로그램의 소리 구간 | 말 속도가 인식 실패의 주된 원인은 아닌 것 같습니다 **[추정]**(눈대중이며 단어별 시간은 재지 않았습니다). |
| 내 녹음은 앞의 약 1.3초가 거의 무음이고, 0.8초 근처에 짧고 넓은 대역의 소리가 있습니다. | **[확인]** | 숨소리·입소리·녹음 시작 소음일 수 있습니다 **[추정]**. Whisper는 무음 뒤 첫 단어를 놓치기 쉬운지는 **[미확인]**입니다. |
| TTS의 무음은 완전히 비어 있고(가장 어두운 색), 내 녹음의 무음에는 낮은 주파수(약 0.3 kHz 이하)에 가는 띠가 남아 있습니다. | **[확인]** 스펙트로그램 아래쪽 | 배경 소음이나 마이크의 저주파 성분일 수 있습니다 **[추정]**. 계획서의 `quality.py` "80 Hz 이하 에너지 비율" 참고 항목이 이런 것을 보려는 것입니다. |
| 모음 구간에 가로 줄무늬(고조파)가 두 소리 모두에 보입니다. TTS는 줄이 곧고 굵으며, 내 녹음은 줄이 위아래로 더 많이 휘어집니다. | **[확인]** 03 STFT 수업의 줄무늬와 같은 현상 | 줄이 휘어진 정도는 음높이(억양) 변화로 볼 수 있습니다 **[일반 지식]**. 내 억양이 TTS보다 더 움직였다는 뜻인지는 **[미확인]**입니다. |
| TTS는 단어 사이에 세로 빈틈이 뚜렷하고, 내 녹음은 1.3~2.0초 구간이 이어져 빈틈이 적습니다. | **[확인]** 세로 빈틈의 선명도 | 단어가 붙어서 읽혔을 가능성이 있습니다 **[추정]**. 이 구간이 `we'll sing`인지는 단어별 시간을 재지 않아 **[미확인]**입니다. |
| 내 녹음은 말하는 동안 4~8 kHz에도 에너지가 넓게 퍼져 있고, TTS는 특정 순간(자음으로 보이는 곳)에만 높습니다. | **[확인]**(두 소리의 최대 진폭을 같게 맞춘 뒤 비교) | 마이크·녹음 환경의 차이일 수 있습니다 **[추정]**. |

**정리**: 두 소리는 눈에 띄게 다르지만, 이 그림만으로는 "인식이 틀린 이유"를 하나로 정할 수 없습니다. 다음에 가장 싸게 확인할 수 있는 것은 두 가지입니다. ① `model.transcribe(..., word_timestamps=True)`로 단어별 시간을 얻어 그림 위에 겹쳐 그리기(어느 구간이 `we'll sing`인지 확인), ② 같은 문장을 앞 무음을 줄이고 다시 녹음해 비교하기.

**색 비교 메모**: 이 그림은 처음에 수업의 `magma`(스펙트로그램)와 `Greens_r`(Log-Mel)를 섞어 썼습니다. 지금은 프로젝트 공통 초록 램프(`theme.CMAP_SPEC`)로 통일했고, 근거는 검토 문서 14.4절에 있습니다.

#### 틀린 구간을 소리 그림 위에 표시하기

S4.5의 그림은 문장 전체를 한 장으로 그린 것이라 **어느 단어가 그림의 어디인지** 알 수 없었습니다. 이번에는 채점 결과와 시간을 연결합니다.

1. 내 녹음과 TTS를 각각 `word_timestamps=True`로 인식해 **단어별 시작·끝 시간**을 얻습니다(`stt.recognize_words`, 앞 무음이 첫 단어에 섞이는 것을 줄이려고 `vad_filter=True`).
2. `scoring`의 결과에서 **연속된 틀린 칸을 한 덩어리**로 묶습니다(`align.error_blocks`). 단어 수가 달라도(`we'll` ↔ `we are`) 한 덩어리로 비교할 수 있습니다.
3. 덩어리마다 정답 단어는 **TTS의 구간**으로, 인식된 단어는 **내 녹음의 구간**으로 바꿔(`align.locate_errors`) 두 그림에 같은 번호 `[k]`로 표시합니다. 한쪽에 단어가 없으면(삭제·삽입) 이웃 단어 사이의 점선 한 줄로 표시합니다.

**읽는 법**: 이 표시는 "내가 말한 소리 중 잘못 인식된 곳이 어디이고, 같은 위치의 기준 소리는 어떻게 생겼는지"를 나란히 보는 용도입니다. **원인을 알려 주지는 않습니다.** 단어 시간은 Whisper가 모델 내부 정보로 **추정한 값**이라 정확한 강제 정렬이 아니며(`tiny`는 오차가 클 수 있음), 구간은 대략의 위치입니다.

In [ ]:
# ── 틀린 구간 표시: 도우미 함수 정의 (실행하면 함수만 만들어집니다) ─────────────────────────
# 역할       : show_error_spans(문장 번호)를 정의합니다. 틀린 단어가 소리의 어디인지 TTS와 내 녹음 그림 위에 같은 번호로 표시합니다.
# 호출 경로    : show_error_spans → flow.prepare(문장·TTS) → flow.locate_errors(인식 2번 → align으로 시간 구간) → report.error_blocks(글) → visualisation.plot_error_spans(그림)
# STT 모델   : tiny, 단어별 시간 모드: stt.recognize_words(내 녹음)와 stt.recognize_words(TTS) 두 번, VAD 켬 (practice.locate_errors 안)
# 가져오는 것   : sentences (S1-1), find_recording (S2), plt·plot_error_spans (소리 그림 준비 셀)
# 만드는 변수   : flow(practice.py 모듈), report(report.py 모듈), show_error_spans → S4의 run_practice도 flow·report를 씁니다
# practice.py: 연습 한 번의 순서를 이어 붙이는 모듈(계산은 하지 않음). 모듈 이름 practice와 겹치지 않게 flow라고 부릅니다.
import practice as flow   # practice.py: 연습 한 번의 순서(문장 → TTS → 점검 → 인식 → 채점 → 틀린 구간). 노트북 함수는 run_practice()라 이름이 겹치지 않지만, 모듈 이름과 구분하려고 flow로 부릅니다.
import report             # report.py: 결과 dict → 읽을 글

def show_error_spans(target, take=1):
    """틀린 덩어리의 시간 구간을 TTS(왼쪽)와 내 녹음(오른쪽)의 파형·스펙트로그램·Log-Mel 위에 같은 번호로 표시한다.
    인식·구간 계산은 flow.locate_errors, 글은 report.error_blocks, 그림은 visualisation.plot_error_spans가 맡는다.
    구간은 Whisper의 단어별 시간(추정값)이라 대략의 위치이고, 평가가 아니라 참고 정보다."""
    # 문장을 찾고 기준 음성(TTS, 캐시)을 준비합니다. prep은 dict: {"id", "text", "lang", "tts_path", ...}
    prep = flow.prepare(target, sentences)
    # 따라읽은 녹음 파일(sNN_takeK.*) 찾기. take 인자로 회차를 고릅니다
    rec = find_recording(prep["id"], take) if prep["id"].isdigit() else None
    if rec is None:
        print("따라읽은 녹음이 없습니다."); return
    # 인식을 두 번 돌립니다(내 녹음, TTS) → 단어별 시간 → 틀린 덩어리의 시간 구간 계산
    out = flow.locate_errors(prep, {"recording": rec})      # 인식이 2번 돌아 몇 초 걸린다
    # 구간을 사람이 읽는 글로 바꿉니다(report.py)
    print(report.error_blocks(prep, out["loc"]))
    # 그림 그리기(visualisation.py). 같은 번호 [k]가 같은 덩어리
    plot_error_spans(out["sounds"], out["loc"]["blocks"],
                     f'문장 #{prep["id"]}: 같은 번호 [k]는 같은 덩어리 (참고 정보, 평가 아님; 구간은 Whisper 단어 시간의 추정값)')
    plt.show()

In [ ]:
# ── 틀린 구간 표시 실행: 3번 문장 ────────────────────────────────────────────
# 역할       : 위에서 정의한 show_error_spans를 3번 문장에 실행합니다(TTS는 캐시, API 0회).
# STT 모델   : tiny, 단어별 시간 모드로 2번 인식 (VAD 켬) — 그래서 몇 초 걸립니다.
# 3번 문장: 내 녹음에서 잘못 인식된 덩어리와, 같은 위치의 TTS 소리를 나란히 봅니다 (TTS는 캐시, API 호출 0회).
# 위쪽 표(구간 숫자)와 그림의 [1] 표시가 같은 곳을 가리키는지 확인하세요. 새 인식이 2번 돌아 몇 초 걸립니다.
show_error_spans(3)

### 실험2) Whisper 모델 크기 비교(같은 녹음, 여러 모델)

면담(11-02)에서 "모델을 올려 보라"는 조언을 받아, **같은 녹음을 모델 크기만 바꿔** 인식·채점해 비교합니다(판단과 근거는 검토 문서 15.2). `we'll sing`처럼 `tiny`가 계속 틀리는 구간이 큰 모델에서 달라지는지 봅니다.

- **비교할 모델은 자동으로 정합니다.** `SIZES = stt.available_models()`가 이 PC의 `models/`에 **이미 내려받아 둔 모델을 전부** 찾아 작은 모델부터 차례로 비교합니다(지금 `tiny`만 있으면 `tiny`만 돕니다). 그래서 기본 실행은 **다운로드도 과금도 없습니다.**
- 비교 대상을 늘리려면 두 가지가 있습니다. ① `DOWNLOAD_MORE = True`: `tiny`·`base`·`small` 중 아직 없는 크기도 내려받아 함께 비교합니다(인터넷이 필요하고 시간이 걸립니다. 받은 모델은 `models/`에 남고 git에서 제외됩니다). ② `SIZES`를 직접 적기: 예 `["tiny", "small", "medium"]`(`medium` 이상은 파일이 훨씬 커서 필요할 때만).
- 인식은 `stt.recognize(..., model_size=)`, 비교와 채점은 `model_compare.compare_models`가 합니다. 이 셀은 표만 보여 줍니다. 모델은 크기마다 한 번만 읽고, 읽는 시간은 인식 시간과 따로 잽니다.
- 한 모델을 읽지 못해도(내려받기 실패 등) 그 모델만 건너뛰고 나머지는 계속합니다(`건너뜀(모델을 읽지 못함)` 줄). 비교가 끝난 모델은 메모리에서 내려(`release=True`) 큰 모델을 여러 개 돌려도 메모리가 쌓이지 않게 합니다.
- 한 사람·몇 개의 녹음 결과라서 "큰 모델이 낫다"고 일반화하지 않습니다. 큰 모델은 틀린 발음을 문맥으로 더 잘 고쳐 적을 수도 있어서, 값은 **인식 일치도의 차이**로만 읽습니다(발음 점수가 아님).
- 30건 녹음 뒤에는 `IDS`, `TAKES`만 늘려 같은 셀로 모델 × 문장 × 회차 표를 만듭니다. 결과를 보고할 때는 아래의 **설정** 줄(장치·계산 방식·beam_size)과 비교한 모델 목록을 함께 적습니다.
- 출력에 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.

In [ ]:
# ── 실험2. Whisper 모델 크기 비교 (같은 녹음, 여러 모델) ──────────────────────────
# 역할       : 같은 녹음을 tiny·base·small 세 모델로 인식해 오류율·시간을 표로 비교합니다.
# 호출 경로    : stt.available_models() → 비교할 모델 크기 목록 → model_compare.compare_models(문장, 크기, 회차)
#               └ 크기마다 stt.load_model(size)로 모델을 읽고, stt.recognize(path, model_size=size)로 인식, scoring.evaluate로 채점
# STT 모델   : SIZES = tiny·base·small (VAD 끔). 여기서만 tiny가 아닌 모델을 읽습니다.
# 인터넷      : DOWNLOAD_MORE=True면 models/에 없는 크기를 처음 한 번 내려받습니다(파일이 큼, API 과금은 아님).
# 범위       : 지금은 문장 1~3의 take1만 (IDS, TAKES). 30건 전체는 아래 '실험3'.
import pandas as pd
from IPython.display import display

import model_compare   # model_compare.py: 같은 녹음을 여러 모델 크기로 인식·채점해 비교(크기별 인식 stt + 채점 scoring)
import stt

# True: tiny·base·small 중 아직 없는 모델을 내려받아 함께 비교 / False: 이미 있는 모델만
DOWNLOAD_MORE = True   # True면 stt.MODEL_SIZES(tiny·base·small) 중 아직 내려받지 않은 크기도 받아서 함께 비교합니다(인터넷 필요, 시간이 걸림, API 과금 없음)
# 비교할 모델 크기 목록(작은 모델부터)
SIZES = stt.available_models(include=stt.MODEL_SIZES if DOWNLOAD_MORE else None)   # 기본: 이 PC의 models/에 이미 있는 모델을 전부(작은 모델부터)
if not SIZES:
    print("models/에 내려받아 둔 모델이 없어 기본 크기(tiny)를 내려받아 씁니다.")
    SIZES = [stt.MODEL_SIZE]
IDS = [1, 2, 3]     # 비교할 문장 번호. 30건 녹음 뒤에는 list(range(1, 11))
TAKES = [1]         # 회차. 30건 녹음 뒤에는 [1, 2, 3]
print("비교할 모델:", SIZES)

# release=True: 모델 하나의 비교가 끝나면 메모리에서 내려, 큰 모델을 여러 개 돌려도 메모리가 쌓이지 않게 합니다.
# 실제 비교가 일어나는 줄. 돌려주는 cmp["rows"]는 모델 × 녹음마다 한 줄 {model, id, take, hyp, wer, S, D, I, N, diff, sec}
cmp = model_compare.compare_models(IDS, SIZES, takes=TAKES, release=True)
print("설정:", {k: v for k, v in cmp["settings"].items() if k != "model"}, "| 모델 읽는 시간(초):", {k: round(v, 1) for k, v in cmp["load_sec"].items()})
for size, err in cmp["errors"].items():
    print("건너뜀(모델을 읽지 못함):", size, "←", err)      # 내려받기 실패 등. 이 모델만 빠지고 나머지는 계속합니다
for m in cmp["missing"]:
    print("건너뜀:", m)

# rows를 표(pandas)로 바꿔 아래에서 보기 좋게 정리합니다
df = pd.DataFrame(cmp["rows"])
if df.empty:
    print("비교할 녹음이 없습니다. data/recordings/sNN_take1.wav를 확인하세요.")
else:
    df["id"] = df["id"].astype(int)
    order = [s for s in SIZES if s in set(df["model"])]       # 표의 열 순서를 SIZES 순서로(작은 모델부터)
    cols = ["model", "vad"] if df["vad"].nunique() > 1 else "model"   # VAD 끔/켬을 둘 다 돌렸을 때만 열을 둘로 나눕니다(안 나누면 두 값이 평균으로 합쳐집니다)
    print("\nWER (오류율: 낮을수록 정답 문장에 가까움)")
    display(df.pivot_table(index=["id", "take"], columns=cols, values="wer")[order].round(3))
    print("인식 시간 (초)")
    display(df.pivot_table(index=["id", "take"], columns=cols, values="sec")[order].round(1))
    print("인식된 문장과 틀린 부분")
    view = df.assign(model=pd.Categorical(df["model"], order, ordered=True))
    display(view[["id", "take", "model", "vad", "hyp", "diff"]].sort_values(["id", "take", "model", "vad"]).reset_index(drop=True))

#### 결과 검토 (tiny·base·small, 문장 1~3의 take1 한 번씩)

표시: **[확인]** 위 출력이나 이 저장소에서 실행해 확인한 값, **[추정]** 해석, **[미확인]** 이 결과만으로는 알 수 없는 것. **한 사람·녹음 3개·1회**의 결과라서 일반화하지 않고 "이 조건에서는 이렇게 나왔다"로만 읽습니다.

**한눈에 보기**

| 문장 | tiny | base | small | 인식된 문장(3번) |
|---|---|---|---|---|
| 1번 | 0.000 | 0.000 | 0.000 | (세 모델 모두 정답과 같음) |
| 2번 | 0.000 | 0.000 | 0.000 | (세 모델 모두 정답과 같음) |
| 3번 | 0.429 | 0.429 | 0.429 | tiny `We are **seeing** together…` / base `We are **singing** together…` / small `We are **staying** together…` |

| 관찰 | 근거 | 해석 |
|---|---|---|
| **1·2번은 세 모델이 모두 맞혔습니다.** 이 두 문장으로는 모델 크기를 비교할 수 없습니다. | **[확인]** WER 0.000 ×6 | 쉬운 문장은 `tiny`로도 충분합니다 **[추정]**. 모델 비교의 변별력은 `we'll sing`처럼 어려운 구간에서 나옵니다. |
| **3번은 모델을 `tiny`→`base`→`small`로 키워도 WER이 0.429로 같고, 틀린 모양도 같습니다.** 세 모델 모두 `we'll`을 `we are`로 들었고, 달라진 것은 `sing` 자리의 단어(seeing / singing / staying)뿐입니다. | **[확인]** 세 줄 모두 `[we'll→we] [+are+] [sing→…]` | `we'll sing`을 못 맞힌 것은 **`tiny`의 용량 문제만은 아닌 것**으로 보입니다 **[추정]**. 앞의 대조 실험(TTS 음성은 `tiny`도 0.000)과 합치면 **내 녹음 쪽 요인**(발음의 연결, 마이크, 앞 무음·소음 등) 가설이 더 힘을 얻습니다 **[추정]**. 어느 요인인지는 여전히 **[미확인]**입니다. |
| **큰 모델이 틀린 발음을 문맥으로 고쳐 적지는 않았습니다.** `We'll sing together until the morning comes`는 문맥만 보면 고치기 쉬운 문장인데도 `base`·`small`은 `we are`를 그대로 적었습니다. | **[확인]** | 걱정했던 "큰 모델이 너무 관대해서 틀린 곳을 못 잡는다"는 문제는 **이 한 사례에서는 나타나지 않았습니다.** 다만 1개 문장이라 큰 모델 전반으로 말할 수는 없습니다. |
| **WER이 같아도 틀린 정도는 다릅니다.** 글자 단위 오류율(CER)은 tiny 0.135(5/37)이고 base·small은 0.162(6/37)입니다. | **[확인]** `scoring.cer`로 계산 | `sing`→`singing`은 의미로는 가까워 보이지만 글자 수로는 `seeing`보다 약간 멉니다. WER·CER 어느 쪽도 "어느 모델이 더 낫다"를 가려 주지 못합니다. **표에 CER 열을 같이 두는 것이 좋습니다.** |
| **속도: 4~5초 녹음(말한 시간은 약 2~3초) 기준으로 `base`는 `tiny`의 약 1.5~3배, `small`은 약 4~6배 걸렸습니다(컴퓨터 부하에 따라 실행마다 달라짐).** (2·3번, 이 실행: tiny 1.3~1.5초, base 3.2~3.7초, small 7.8~8.4초) | **[확인]** 인식 시간 표 | 이 PC(CPU, int8)에서는 `small`이 4~5초짜리 녹음 한 개에 5~8초 걸립니다. 웹 화면(Streamlit)에서 바로 결과를 보여 주기에는 무거울 수 있습니다 **[추정]**. |
| 1번 문장의 인식 시간(tiny 4.3초, base 4.4초)과 모델 읽는 시간(base 8.7초, small 22.0초)은 **비교에 쓰지 않습니다.** | **[확인]** | 첫 호출의 준비 시간과, 이번에 처음 **내려받은** 시간이 섞였을 가능성이 큽니다 **[추정]**. 속도는 2·3번 값으로 비교했습니다. |

**추가 확인 (위 표 밖에서 로컬로 돌려 본 값) — 모델 크기만이 아니라 인식 옵션도 결과를 바꿉니다**

위 표는 모든 모델에 **같은 옵션(`vad_filter` 끔)** 으로 돌린 것입니다. 같은 녹음을 `vad_filter=True`(말소리가 없는 구간을 걸러낸 뒤 인식)로 다시 돌려 보았습니다 **[확인]**.

| 문장 | tiny (VAD 끔 → 켬) | base (끔 → 켬) | small (끔 → 켬) |
|---|---|---|---|
| 1번 | 0.000 → 0.000 | 0.000 → 0.000 | 0.000 → 0.000 |
| 2번 | 0.000 → 0.000 | 0.000 → 0.000 | 0.000 → 0.000 |
| 3번 | 0.429 → 0.429 | 0.429 → 0.429 | **0.429 → 0.286** (`We will sing together…`) |

- **`small`만, 3번에서만** 결과가 달라졌습니다. VAD를 켜면 `we are staying`이 `we will sing`이 되어 오류가 3개에서 2개로 줄었습니다(치환 2·삽입 1 → 치환 1·삽입 1, 프로젝트의 표시로는 `[+we+] [we'll→will]`, 2/7). 인식된 단어 수는 두 경우 모두 8개입니다. 다른 문장은 그대로여서 **악화 사례는 없었습니다.** 단어 시간 옵션(`word_timestamps`)만 켠 경우는 달라지지 않았습니다.
- 단어별 확신도(`prob`)를 보면 불확실한 곳이 보입니다 **[확인]**: base `singing` **0.19**(낮음), small(VAD 켬) `will` 0.34·`sing` 0.76. 모델이 헷갈린 자리와 틀린 자리가 겹칩니다 **[추정]**.
- 이 녹음은 앞 무음이 0.8초이고 4.8초 중 말한 시간이 2.8초입니다(`quality` 점검 결과). 앞의 무음·소음이 `small`의 디코딩에 영향을 줬을 수 있습니다 **[추정]**. 녹음 한 개·한 번의 결과라서 우연일 수도 있습니다 **[미확인]**.
- **파이프라인에 주는 의미 [추정]**: 지금 채점(`practice.read`)은 VAD 없이(`stt.recognize`), 틀린 구간 표시(`practice.locate_errors`)는 VAD를 켜고(`stt.recognize_words`) 인식합니다. 기본 모델 `tiny`에서는 둘이 같지만, **`small` 이상으로 바꾸면 채점된 문장과 구간 표시의 문장이 서로 달라질 수 있습니다.** 기본 모델을 바꾸기 전에 두 경로의 옵션을 같게 맞춰야 합니다.

**결론과 다음에 할 일**

1. **기본 모델은 `tiny`를 유지합니다.** 이 3개 녹음에서는 `base`·`small`이 WER을 개선하지 못했고 약 1.5~6배 느렸습니다. 결정은 30건 녹음 뒤에 다시 합니다.
2. **30건 녹음 뒤에 같은 표를 확장합니다**(`IDS = list(range(1, 11))`, `TAKES = [1, 2, 3]`). 이때 **VAD 끔/켬을 변수로** 비교합니다(`compare_models(..., vad_filters=(False, True))`로 이미 할 수 있고, 표는 `columns=["model", "vad"]`로 펼칩니다), 표에 **CER 열**도 같이 둡니다.
3. **`we'll`의 원인 분리는 모델이 아니라 녹음 쪽에서 합니다.** 같은 문장을 (a) 앞 무음을 줄여 다시 녹음, (b) 일부러 또렷하게 `we'll`과 `we will`을 각각 읽어 비교, (c) 다른 마이크로 녹음해 봅니다.
4. 출력의 `HF_TOKEN` 경고(`unauthenticated requests`)는 Hugging Face에서 내려받을 때의 속도 제한 안내이며 동작에는 영향이 없습니다. 모델을 모두 내려받은 뒤에는 나오지 않습니다 **[일반 지식]**.

### 실험3) 녹음 30건(10문장 × 3회) 일괄 인식·채점 (정규화 방침 C)

`data/recordings/`의 30개 파일을 `tiny`로 모두 인식하고, **같은 인식 결과를 두 가지 규칙으로 채점**합니다(판단과 근거는 검토 문서 13.4).

- **통일(기본)**: 정답과 인식 결과 양쪽에서 축약형을 펼치고(`we're`→`we are`) 0~99의 숫자 단어를 숫자로 바꿉니다(`twenty`→`20`). 소리는 같은데 표기만 다른 것을 오류로 세지 않습니다.
- **엄격**: `scoring.wer(..., unify=False)`. 표기 차이도 오류로 셉니다. 두 값의 차이가 "표기 때문에 생긴 오류"의 크기입니다.
- 로컬 Whisper만 쓰므로 **API 과금이 없고**, 이 PC(CPU)에서 약 1분 걸립니다. 노트북에서 직접 녹음하지 않고 `scripts/record_takes.py`로 만든 파일을 읽습니다.

**이 실험의 목적과 순서 (원인 → 결과)**: 정규화 방침을 "세우려는" 실험도, 방침을 "처음 검증하는" 실험도 아닙니다. **방침을 세운 뒤에 하는 효과·부작용 확인이자 30건의 기준선 측정**입니다.

1. **요건**: L1에는 "정규화 방침 확정(#3)"과 "30건 통합·검증(#10)"이 둘 다 있습니다. 채점 규칙이 바뀌면 점수가 바뀌므로 **규칙을 먼저 정한 뒤에** 30건을 채점해야 의미가 있습니다.
2. **방침의 근거 두 가지**: (가) 설계 논리: 소리는 같은데 표기만 다른 것(`we're`/`we are`, `twenty`/`20`)을 오류로 세면 점수가 "인식 일치도"에서 멀어집니다. 문장 7은 숫자 규칙을 시험하려고 일부러 넣은 문장입니다. (나) 사전 실행: 규칙을 정하기 전에 30건을 엄격 규칙으로 한 번 돌려 보니 문장 7의 오류 3개가 전부 표기 차이였습니다(AI 어시스턴트가 터미널에서 돌린 값이라 이 노트북에는 기록이 없었음). 이 관찰과 옵션 비교(A 엄격 유지 / B 통일 / C 통일 기본 + 엄격 선택)로 **C안**을 정했습니다.
3. **구현**: `scoring.normalize(unify=True)`와 단위 테스트(`Unify`).
4. **이 실험**: 구현한 방침으로 **같은 30건을 두 규칙으로** 채점합니다. 목적은 셋입니다. ⓐ **기록과 재현**: 사전 실행 값이 내 화면에 없었으므로 직접 볼 수 있는 곳에 남기고 같은 값이 나오는지 확인합니다. ⓑ **효과와 부작용 측정**: 엄격과 통일의 차이를 분해합니다. ⓒ **L1 기준선**: 처리 오류·상태 경고·회차 변동을 보고, 손계산 5건 후보를 고릅니다.
5. **한계**: 방침의 근거가 된 데이터(문장 7)와 이 실험의 데이터가 **같은 30건**입니다. 그래서 이 실험은 방침의 *독립적인 검증*이 아니라 *적용 결과의 확인*입니다. 방침은 이 데이터에 맞춰 조정한 값이 없는 규칙(축약형·숫자 목록)이지만, "이 규칙이 옳은가"를 새로 확인하려면 다른 문장·녹음이 필요합니다. 손계산 5건도 "프로그램이 규칙대로 계산하는가"를 보는 것이지 "규칙이 옳은가"를 보는 것은 아닙니다.

**표를 읽는 법**
1. **[1] 문장 × 회차 표**: 같은 문장을 세 번 읽었을 때 오류율이 얼마나 흔들리는지(범위 = 최댓값 − 최솟값)를 봅니다. 하나의 평균으로 합치지 않고 이 표를 먼저 봅니다.
2. **[2] 전체 요약**: **Micro**(오류 합 ÷ 정답 단어 합)를 기본으로, Macro(문장별 평균)를 보조로 봅니다. 통일은 `can't`→`can not`처럼 단어가 늘어서 **분모(정답 단어 합)가 엄격과 다릅니다.** 그래서 같은 오류도 점수가 조금 달라지므로, 값을 말할 때는 어느 쪽인지 함께 적습니다.
3. **[3] 틀린 건 상세**: 인식된 문장과 `[정답→인식]` 표시를 봅니다. **손계산 5건 대조는 여기서 고릅니다.** 통일 기준의 S·D·I·N을 직접 세어 표의 값과 같은지 비교합니다.

**주의**: 점수는 발음 점수가 아니라 문장 인식 일치도입니다. 화자는 본인 1명, 마이크·환경은 하나라서 Whisper의 일반 정확도를 말하는 결과가 아닙니다. 이 셀의 출력에는 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.

In [ ]:
# ── 실험3. 녹음 30건(10문장 × 3회) 일괄 인식·채점 ───────────────────────────────
# 역할       : data/recordings/의 30개 파일을 모두 인식하고, 같은 결과를 '통일'과 '엄격' 두 규칙으로 채점해 표 3개로 보여 줍니다.
# 호출 경로    : model_compare.compare_models([tiny], 문장 1~10, 회차 1~3) → 파일마다 stt.recognize → 인식 문장
#            → scoring.wer(통일) / scoring.wer(unify=False, 엄격) / quality.check_file(녹음 상태 경고)
# STT 모델   : tiny (stt.MODEL_SIZE), CPU·int8·beam_size=1, VAD 끔. 모델은 한 번만 읽고 30건에 재사용
# 과금       : 없음 (로컬). 약 1분
# 만드는 변수   : df30 — 30건의 결과 표 (손계산 5건 대조에 사용)
# 실험3) 녹음 30건(10문장 × 3회) 일괄 인식·채점. 로컬 Whisper라 API 과금 없음, CPU에서 약 1분.
import pandas as pd
from IPython.display import display

import model_compare   # 같은 녹음을 모델 크기별로 인식·채점하는 도구(여기서는 기본 모델 하나만 씀)
import quality
import scoring
import stt

ALL_IDS = list(range(1, 11))
ALL_TAKES = [1, 2, 3]
# 30건 인식이 도는 줄. 돌려주는 res["rows"]는 파일마다 한 줄(hyp=인식 문장 등)
res = model_compare.compare_models(ALL_IDS, sizes=[stt.MODEL_SIZE], takes=ALL_TAKES)
print("인식 설정:", res["settings"])
print("못 찾은 녹음:", res["missing"] or "없음", "| 모델 읽기 실패:", res["errors"] or "없음")

records = []
for r in res["rows"]:
    # 통일 채점: 정답과 인식 모두 축약형(we're→we are)·숫자(twenty→20)를 같은 규칙으로 맞춥니다 (scoring.normalize의 unify=True가 기본)
    uni = scoring.wer(r["ref"], r["hyp"])                    # 표기 통일(기본, 방침 C): we're = we are, twenty = 20
    # 엄격 채점: unify=False로 표기 차이도 오류로 셉니다
    strict = scoring.wer(r["ref"], r["hyp"], unify=False)    # 엄격: 표기 차이도 오류로 센다
    # 녹음 상태 점검(음량·무음·클리핑·말한 시간). 경고가 있어도 인식·채점은 그대로 진행합니다
    warns = quality.check_file(stt.find_recording(int(r["id"]), r["take"]))["warnings"]
    records.append({
        "문장": int(r["id"]), "회차": r["take"], "파일": r["file"], "인식 문장": r["hyp"],
        "WER 통일": uni["rate"], "S": uni["S"], "D": uni["D"], "I": uni["I"], "N": uni["N"],
        "WER 엄격": strict["rate"], "오류 엄격": strict["S"] + strict["D"] + strict["I"], "N 엄격": strict["N"],
        "틀린 부분(통일)": scoring.format_diff(uni["ops"]), "틀린 부분(엄격)": scoring.format_diff(strict["ops"]),
        "상태 경고 수": len(warns), "인식 초": round(r["sec"], 1),
    })
df30 = pd.DataFrame(records)       # 이 표를 아래 셀과 손계산 대조에 다시 쓸 수 있게 변수로 남겨 둡니다.

# 1) 문장 × 회차 표: 통일 / 엄격 WER, 범위(최댓값 − 최솟값)
wide = pd.concat({"통일": df30.pivot(index="문장", columns="회차", values="WER 통일"),
                  "엄격": df30.pivot(index="문장", columns="회차", values="WER 엄격")}, axis=1)
wide[("통일", "범위")] = wide["통일"].max(axis=1) - wide["통일"].min(axis=1)
wide[("엄격", "범위")] = wide["엄격"].max(axis=1) - wide["엄격"].min(axis=1)
print("\n[1] 문장 × 회차 WER (통일 = 기본, 엄격 = 표기 차이도 오류)")
display(wide.round(3))

# 2) 전체: Micro(오류 합 ÷ 정답 단어 합)를 기본, Macro(문장별 평균)를 보조로 적습니다.
err_uni = (df30["S"] + df30["D"] + df30["I"]).sum()
summary = pd.DataFrame({
    "Micro": [err_uni / df30["N"].sum(), df30["오류 엄격"].sum() / df30["N 엄격"].sum()],
    "Macro": [df30.groupby("문장")["WER 통일"].mean().mean(), df30.groupby("문장")["WER 엄격"].mean().mean()],
    "오류 합": [err_uni, df30["오류 엄격"].sum()],
    "정답 단어 합": [df30["N"].sum(), df30["N 엄격"].sum()],
    "WER 0인 건": [(df30["WER 통일"] == 0).sum(), (df30["WER 엄격"] == 0).sum()],
}, index=["통일(기본)", "엄격"])
print("[2] 전체 요약 (녹음 %d건)" % len(df30))
display(summary.round(3))
print("녹음 상태 경고가 있는 건:", int((df30["상태 경고 수"] > 0).sum()), "건 / 인식 시간 평균", round(df30["인식 초"].mean(), 1), "초")

# 3) 틀린 건 상세: 손계산 5건은 여기서 고릅니다(통일 기준 S·D·I·N으로 직접 계산해 비교).
wrong = df30[(df30["WER 통일"] > 0) | (df30["WER 엄격"] > 0)]
print("\n[3] 한 번이라도 틀린 건 (%d건)" % len(wrong))
with pd.option_context("display.max_colwidth", None, "display.width", 220):
    display(wrong[["문장", "회차", "WER 엄격", "WER 통일", "S", "D", "I", "N", "인식 문장", "틀린 부분(통일)", "틀린 부분(엄격)"]]
            .round(3).reset_index(drop=True))


#### 결과 검토 (실험3, `tiny`, 정규화 방침 C)

**한 줄 결론 (이 30건, 이 조건에서)**
- 표기 통일은 문장 7의 표기 오류(3회 × 3개 = 9개)를 없애고, 진짜 오인식(문장 3, 6, 8, 10)은 그대로 남깁니다.
- 진짜 오인식은 **같은 문장도 회차마다 달라집니다.** 한 번의 값으로 그 문장의 정확도를 말할 수 없습니다.
- 통일 0.044와 엄격 0.092는 표기 처리 방식에 따라 점수가 두 배 가까이 달라진다는 뜻이므로 **어느 쪽인지 항상 함께 적습니다.**

**표를 이렇게 읽습니다**
- **표 [2] (전체 요약)**: 정답 단어 100개당 틀린 단어는 통일 기준 약 **4.4개**(11/252), 엄격 기준 약 **9.2개**(22/240)입니다. 오류가 0인 건은 30건 중 통일 **23건**, 엄격 **20건**입니다.
- **두 값이 다른 이유**: 엄격 22개가 통일 11개로 줄어든 차이 11개는 ① 문장 7의 표기 오류 9개(3회 × 3개: `We're`, `are`, `twenty`)가 사라진 것과 ② 문장 3에서 `we'll`이 `we will`로 펼쳐져 `We`가 일치로 잡히면서 회차당 1개씩 줄어든 2개(아래 부작용 1)의 합입니다 **[확인: 합계 계산]**. 발음 문제가 아니라 표기와 계산 방식의 차이입니다.
- **표 [1] (문장 × 회차)**: 행은 문장, 열은 회차, 값은 오류율입니다. 문장 3, 6, 8, 10은 같은 문장을 읽어도 회차마다 값이 다릅니다. 예를 들어 문장 3은 1·2회차에서 틀리고 3회차에서 맞았습니다.
- **표 [3] (틀린 건 상세)**: 어디가 어떻게 틀렸는지 `[정답→인식]`으로 봅니다. 손계산 5건은 여기서 고릅니다.
- **주의**: 이 점수는 발음 점수가 아니라 문장 인식 일치도이고, 화자 1명·마이크 1개의 30건입니다.

---

**참고 baseline** (AI 어시스턴트가 터미널에서 먼저 실행한 값. 같은 값이 내 노트북 실행에서 재현됨, 아래 "노트북 재현" 참고; `tiny`, 30건, 처리 오류 0건, 입력 상태 경고 0건)

| 문장 | 엄격 take1·2·3 | 통일 take1·2·3 | 인식된 문장 (오류가 있는 회차) |
|---|---|---|---|
| 3 | 0.429, 0.429, 0 | 0.250, 0.250, 0 | `We are seeing together…` |
| 6 | 0, 0.143, 0.143 | 0, 0.125, 0.125 | `…about this summer` |
| 7 | 0.300 ×3 | **0 ×3** | `We are going… 20 stars…` |
| 8 | 0, 0.083, 0.083 | 0, 0.083, 0.083 | `in the lamps` / `under their lamps` |
| 10 | 0, 0.429, 0 | 0, 0.429, 0 | `is a prize us` |
| 1·2·4·5·9 | 모두 0 | 모두 0 | |

- **전체(Micro = 오류 합 ÷ 정답 단어 합)**: 엄격 22/240 = **0.092**, 통일 11/252 = **0.044**(분모가 다른 이유는 아래 부작용 1). Macro(문장별 평균): 0.088 → 0.045. WER 0인 건: 20건 → **23건**. 차이 11개 오류가 모두 표기 때문인 것은 아니고, 아래 부작용이 섞여 있습니다.
- **부작용 1 — 분모 N이 바뀝니다.** `can't`가 `can not`이 되어 단어 수가 늘므로 같은 1개 오류라도 문장 6은 1/7 = 0.143 → 1/8 = 0.125가 됩니다. 문장 3도 `we'll` 때문에 N이 7 → 8이 되고, `We`가 일치로 잡혀 S=3 → 2로 줄었습니다(0.429 → 0.250). 같은 오류의 점수가 규칙에 따라 달라지므로 **결과를 보고할 때 엄격/통일 중 어느 쪽인지 반드시 적습니다.**
- **부작용 2 — 틀린 부분 표시가 달라집니다.** 화면에는 `we'll` 대신 `we will`이 보이고, 소리 그림의 틀린 구간이 더 정확해집니다(문장 3: `We'll sing` 전체 → `are seeing`만, 1.59~1.97초) **[실행 확인, `test_align`]**. 한 단어가 두 토큰이 되면 두 토큰이 같은 단어 시간을 공유합니다.
- **부작용 3 — 표기를 통일하므로 "발음 점수"와 더 멀어지지는 않지만**, `can't`와 `can`처럼 소리가 다른 경우까지 가리는 것은 아닙니다. `can't → can not`은 그대로 `not`이 남아 구분됩니다 **[실행 확인]**.

**노트북 재현과 문장별 검토 (11-03, 내 실행: `practice.ipynb` 실험3, `tiny`·CPU·int8·beam_size=1·`en`)**

- **재현 [확인]**: 위 참고 baseline과 같은 값이 노트북에서 그대로 나왔습니다(Micro 통일 0.044 / 엄격 0.092, WER 0인 건 23 / 20, 못 찾은 녹음 0, 모델 읽기 실패 0, **녹음 상태 경고 0건**, 인식 시간 평균 1.0초). 같은 파일을 두 번 인식해도 인식 문장이 같았습니다(두 번만 비교). **11-03 밤 정정**: 처음에는 그 이유를 `beam_size=1`이라 결정적이기 때문이라고 추정했습니다. 그런데 노래 구간에서는 같은 설정으로도 결과가 실행마다 달라지는 것을 확인했습니다(원인: faster-whisper의 온도 폴백, 검토 문서 13.5). 이 30건에서 두 번 같았던 것은 깨끗한 말소리라 폴백이 걸리지 않았기 때문일 수 있습니다 **[추정]**. 30건 전체를 여러 번 반복해 확인한 것은 아닙니다 **[미확인]**.
- **오류가 없는 문장**: 1, 2, 4, 5, 9번은 3회 모두 WER 0(15건). `don't`, `hear/here`, `right/light`, `along/lake`는 이 녹음에서 문제가 되지 않았습니다. 7번은 통일 규칙에서 3회 모두 0.

| 문장 | 회차별 통일 WER | 틀린 모양 | 관찰 |
|---|---|---|---|
| 3 | 0.250, 0.250, **0** | take1·2 `We are seeing` (S2: will→are, sing→seeing) | **같은 사람·같은 문장인데 3번째만 맞음.** 통일 규칙은 `we'll`≠`we are`로 둡니다(소리가 다르므로 맞게 둠) |
| 6 | **0**, 0.125, 0.125 | take2·3 `this summer` (that→this) | 1번째만 맞음. 원인은 **[미확인]** |
| 8 | **0**, 0.083, 0.083 | take2 `under→in`, take3 `the→their` | 두 회차의 틀린 곳이 **서로 다름**(한 단어씩) |
| 10 | **0**, 0.429, **0** | take2 `is a prize us` (S2·I1) | 단어 경계 오류(`surprise`가 `a prize`로 갈라짐). 나머지 두 번은 정답 |
| 7 | 0 ×3 (엄격 0.300 ×3) | `We are … 20 stars` | 표기 차이만. 3회 모두 같은 방식이라 규칙으로 해결되는 유형 |

- **관찰 [확인]**: 오류가 있는 4개 문장(3, 6, 8, 10) 모두 **회차 사이에 결과가 달랐습니다**(범위 0.083~0.429). 같은 사람·같은 마이크·같은 환경에서도 한 번 읽은 결과만으로 그 문장의 "정확도"를 말하기는 어렵습니다. 그래서 문장 × 회차 표를 먼저 보고 평균 하나로 합치지 않는 방침(13.2)이 맞았습니다.
- **원인은 확정하지 않습니다 [미확인]**: 어느 회차가 왜 틀렸는지는 이 표만으로 알 수 없습니다. 상태 점검 경고가 0건이라 음량·무음·클리핑·길이 문제는 아니라는 것까지만 말할 수 있습니다. 발음·억양·말 속도·Whisper `tiny`의 한계 중 무엇인지는 단어별 확신도(`prob`)나 큰 모델 비교(15.2의 ③)로 이어서 봅니다.
- **일반화 금지**: 화자 1명, 마이크 1개, 오류가 있는 녹음 7건(통일 기준). 위 값은 "이 조건의 이 30건"에 대한 결과입니다.

### 실험4) 따라읽기 30건: VAD 끔/켬 × 모델 크기(tiny·base) 비교와 VAD 기본값 결정 (이슈 #34)

**VAD**는 말소리가 없는 구간을 걸러 내고 Whisper에 말소리만 넘기는 기능입니다. 11-03 면담에서 강사님이 "무음까지 학습해 문제가 될 수 있으니 음성만 넘기는 편이 정확도를 높일 수 있을 것"이라고 하셨고, 결과를 보고 정하라고 하셨습니다. 노래에서는 원곡에 VAD를 켜면 가사를 통째로 지웠지만(검토 문서 13.5), **말소리(내 따라읽기)에서는 다를 수 있어 따로 확인**합니다. 지금은 `stt.recognize`가 VAD 끔, `recognize_words`가 VAD 켬이 기본이라 **두 경로의 설정이 서로 다른 것**도 이 실험으로 정합니다.

**실험 설계**
- 녹음 30건(10문장 × 3회), `tiny`·`base` × VAD 끔/켬, **기본 설정**(온도 폴백 켬), 같은 조건 전체를 **3회 되풀이** = 360번 인식. 로컬 Whisper라 **API 과금이 없고**, CPU에서 약 10분 안팎입니다 **[추정]**. 인식하는 동안 다른 무거운 작업을 돌리지 않습니다(인식 시간 비교가 흔들립니다).
- 도구는 `model_compare.compare_models(vad_filters=(False, True))`입니다. 점수는 **표기 통일(기본)** 과 **엄격** 둘 다 적습니다(검토 문서 13.4). 값을 말할 때는 어느 쪽인지 함께 적습니다.
- 오류율(WER)은 **발음 점수가 아니라 문장 인식 일치도**입니다. 화자는 본인 1명, 마이크·환경 하나라서 일반화하지 않습니다.

**판정 기준 (결과를 보기 전에 정함, 2026-11-04)**
1. **차이가 있는가**: 모델별로 VAD 켬과 끔의 **오류 단어 수(3회 평균)가 3개 이상 차이 나고 3회 범위(micro 최솟값~최댓값)가 겹치지 않을 때만** "차이 있음"으로 봅니다. 3개는 "한 문장에서 한두 단어가 바뀐 것은 차이로 보지 않는다"는 간이 기준입니다 **[추정]**. 반복 결과가 모두 같아 범위의 폭이 0이어도 이 3개 기준 때문에 작은 차이를 결론에 쓰지 않습니다(노래 검증 시험에서 폭 0 때문에 기준이 모호했던 점의 보완, 절차 문서 12.2).
2. **빈 인식**: VAD 켬에서 인식 결과가 **빈 문장인 건이 1건이라도 있으면** 그 모델에서는 켬을 위험한 설정으로 봅니다(노래에서 가사를 지운 것과 같은 현상).
3. **기본값 결정 규칙**
   - (가) 켬이 **차이 있음으로 더 낫고**, 빈 인식이 0건이며, `tiny`·`base` **둘 다** 같은 방향이면 → 두 경로 모두 **켬**으로 통일합니다.
   - (나) 끔이 더 낫거나 켬에서 빈 인식이 있으면 → 두 경로 모두 **끔**으로 통일합니다(`recognize_words`의 기본을 끔으로).
   - (다) **구분 불가**(차이 있음 기준에 못 미침)이거나 모델마다 방향이 다르면 → **끔**으로 통일합니다. 이유: 끔이 7강 설정과 같고, 노래에서 켬이 위험했으며, 말소리에서도 이득이 확인되지 않으면 바꿀 근거가 없기 때문입니다 **[추정]**.
4. **한계**: 30건은 문장 10개를 한 사람이 읽은 것이고 `tiny`·`base`만 봅니다. `small`·`medium`과 VAD의 관계는 이 실험에 넣지 않았습니다(문장 1~3에서 small + VAD 켬이 한 문장만 좋아졌던 값은 검토 문서 15.2).

**이 셀의 출력에는 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.**

In [ ]:
# ── 실험4. 따라읽기 30건: VAD 끔/켬 × 모델 크기(tiny·base) × 3회 (이슈 #34) ──────────────────────────
# 역할       : 같은 30건 녹음을 tiny·base 모델에서 VAD 끔/켬으로 3번씩 인식·채점해, VAD 기본값을 정할 표를 만듭니다.
# 호출 경로    : model_compare.compare_models(문장 1~10, 크기, 회차 1~3, vad_filters=(False, True)) × REPEATS회
#               └ 파일마다 stt.recognize(vad_filter=...) → scoring.evaluate(통일) / scoring.wer(unify=False, 엄격)
# STT 모델   : tiny·base, CPU·int8·beam_size=1, 온도 폴백 켬(기본 설정). VAD 끔/켬만 바꿉니다.
# 과금       : 없음(로컬). 약 10분 안팎 [추정]. 실행 중 다른 무거운 작업은 피합니다.
# 만드는 변수   : df4 — 360번 인식의 한 줄씩 결과 표(model, vad, run, 문장, 회차, 인식 문장, 오류 수, 빈 인식 여부, 초)
import time

import pandas as pd
from IPython.display import display

import model_compare   # 같은 녹음을 여러 모델·VAD 설정으로 인식·채점하는 도구
import scoring
import stt

SIZES = ["tiny", "base"]      # 비교할 모델 크기(둘 다 models/에 있어야 합니다)
VADS = (False, True)          # VAD 끔 / 켬
REPEATS = 3                   # 같은 조건 전체를 되풀이할 횟수(흔들림 범위를 보려고)
IDS = list(range(1, 11))      # 문장 1~10
TAKES = [1, 2, 3]             # 회차 1~3 → 파일 30개
missing_models = [s for s in SIZES if s not in stt.available_models()]
assert not missing_models, f"models/에 없는 모델: {missing_models} (먼저 내려받아 주세요)"

rows, t_all = [], time.time()
for run in range(1, REPEATS + 1):
    t0 = time.time()
    res = model_compare.compare_models(IDS, SIZES, takes=TAKES, vad_filters=VADS)   # 모델은 한 번 읽으면 재사용됩니다
    for r in res["rows"]:
        strict = scoring.wer(r["ref"], r["hyp"], unify=False)     # 엄격 채점: 표기 차이도 오류로 셉니다
        rows.append({**r, "run": run, "err": r["S"] + r["D"] + r["I"],
                     "err_strict": strict["S"] + strict["D"] + strict["I"], "N_strict": strict["N"],
                     "empty": r["hyp"].strip() == ""})
    print(f"반복 {run}/{REPEATS} 완료 ({time.time() - t0:.0f}초, 누적 {time.time() - t_all:.0f}초)", flush=True)
print("인식 설정:", res["settings"], "| 못 찾은 녹음:", res["missing"] or "없음", "| 모델 읽기 실패:", res["errors"] or "없음")
df4 = pd.DataFrame(rows)
df4["vad"] = df4["vad"].map({False: "끔", True: "켬"})

# [1] 조건별 요약: 반복마다 micro(오류 합 ÷ 정답 단어 합)를 구해 평균·최솟값·최댓값을 냅니다
per_run = (df4.groupby(["model", "vad", "run"])
              .agg(err=("err", "sum"), N=("N", "sum"), err_strict=("err_strict", "sum"), N_strict=("N_strict", "sum"),
                   empty=("empty", "sum"), sec=("sec", "mean")).reset_index())
per_run["micro"] = per_run["err"] / per_run["N"]
per_run["micro_strict"] = per_run["err_strict"] / per_run["N_strict"]
summary = (per_run.groupby(["model", "vad"])
                  .agg(runs=("run", "count"), N=("N", "first"), 오류합_평균=("err", "mean"),
                       micro_통일=("micro", "mean"), 통일_min=("micro", "min"), 통일_max=("micro", "max"),
                       micro_엄격=("micro_strict", "mean"), 빈인식_최대=("empty", "max"), 인식초_평균=("sec", "mean"))
                  .reset_index())
summary["model"] = pd.Categorical(summary["model"], SIZES, ordered=True)
summary = summary.sort_values(["model", "vad"]).reset_index(drop=True)
print("\n[1] 조건별 요약 (micro = 오류 합 ÷ 정답 단어 합, min~max = 반복 3회 범위, 빈인식 = 한 반복에서 빈 문장이 나온 건수의 최대)")
display(summary.round(3))

# [2] 사전 판정 기준 1·2를 그대로 적용(기계적 계산, 해석과 최종 결정은 아래 마크다운 셀에서)
print("\n[2] 사전 판정 기준 적용 (차이 있음 = 오류 단어 3개 이상 차이 + 범위가 겹치지 않음)")
for m in SIZES:
    off = summary[(summary["model"] == m) & (summary["vad"] == "끔")].iloc[0]
    on = summary[(summary["model"] == m) & (summary["vad"] == "켬")].iloc[0]
    diff = off["오류합_평균"] - on["오류합_평균"]              # 양수면 켬이 오류가 적음
    overlap = not (on["통일_max"] < off["통일_min"] or off["통일_max"] < on["통일_min"])
    if abs(diff) >= 3 and not overlap:
        verdict = "켬이 더 낫다" if diff > 0 else "끔이 더 낫다"
    else:
        verdict = "구분 불가"
    print(f"  {m}: 오류 평균 끔 {off['오류합_평균']:.1f} / 켬 {on['오류합_평균']:.1f} (끔−켬 = {diff:+.1f}), 범위 겹침 {overlap}, "
          f"켬 빈 인식 최대 {int(on['빈인식_최대'])}건 → {verdict}")

# [3] 문장 × 조건 WER(3회 평균): 어느 문장에서 VAD가 영향을 주는지
print("\n[3] 문장 × 조건 WER (통일, 3회 × 회차 평균)")
display(df4.pivot_table(index="id", columns=["model", "vad"], values="wer").reindex(columns=pd.MultiIndex.from_product([SIZES, ["끔", "켬"]])).round(3))

# [4] VAD 끔/켬에서 인식 문장이 달라진 건(같은 모델·반복·문장·회차끼리 비교)
key = ["model", "run", "id", "take"]
wide = df4.pivot_table(index=key, columns="vad", values="hyp", aggfunc="first").reset_index()
changed = wide[wide["끔"] != wide["켬"]]
print(f"\n[4] VAD 끔/켬에서 인식 문장이 달라진 건: {len(changed)} / {len(wide)} (모델 × 반복 × 녹음)")
if len(changed):
    display(changed.rename(columns={"끔": "끔 인식", "켬": "켬 인식"}).head(30))

#### 결과 검토 (실험4, 따라읽기 30건 × VAD 끔/켬 × tiny·base × 3회)

**먼저 읽는 법.** 위 [1]이 조건별 요약, [2]가 사전 판정 기준을 기계적으로 적용한 결과, [3]이 문장별 WER, [4]가 VAD 끔/켬에서 인식 문장이 달라진 건입니다. 점수는 발음 점수가 아니라 **문장 인식 일치도**이고, 값은 표기 통일 기준(엄격 기준은 [1]의 `micro_엄격`)입니다. 정답 단어 합 N은 252입니다.

**핵심 숫자 [확인: 위 출력]**

| 모델 | VAD | 오류 단어 수(3회 평균) | micro WER(통일) | 3회 범위 | 빈 인식(최대) | 인식 시간(초/건) |
|---|---|---|---|---|---|---|
| tiny | 끔 | 11 | 0.044 | 0.044 ~ 0.044 | 0 | 1.33 |
| tiny | 켬 | **8** | 0.032 | 0.032 ~ 0.032 | 0 | 1.47 |
| base | 끔 | **6** | 0.024 | 0.024 ~ 0.024 | 0 | 1.96 |
| base | 켬 | 7 | 0.028 | 0.028 ~ 0.028 | 0 | 2.12 |

**1. 사전 기준 적용 (기준은 결과를 보기 전에 위에 적어 둠)**
- **기준 1 (차이 있음)**: tiny는 오류가 11 → 8로 **3개 줄어** 기준(3개 이상 + 범위가 안 겹침)을 **딱 맞게** 넘었고, base는 6 → 7로 **1개 늘어** 구분 불가입니다.
- **기준 2 (빈 인식)**: VAD 켬에서 빈 문장은 **0건**입니다(30건 × 3회 × 2모델). 노래에서 가사를 통째로 지운 일은 말소리에서 **일어나지 않았습니다**.
- **기준 3 (기본값)**: 켬이 낫다는 방향이 `tiny`·`base`에서 **갈립니다**(tiny는 켬이 낫고 base는 끔이 낫거나 같음) → **(다)에 해당 → 끔**.

**2. 차이의 정체: 녹음 1개씩입니다 [확인: 위 [4]와 내가 같은 두 녹음만 다시 인식한 참고 값]**
- **tiny의 차이 3개는 전부 문장 10 take2 한 녹음**에서 나왔습니다. VAD 끔은 `will→is`, `+a+`, `surprise→prize`로 3개를 틀렸고, 켬은 모두 맞혔습니다. 나머지 29건은 끔과 켬이 같았습니다. 반복 3회가 모두 같은 값이라 **독립된 증거가 3개가 아니라 같은 한 녹음이 3번 재현된 것**입니다.
- **base의 차이 1개도 한 녹음**(문장 8 take3)입니다. 끔은 맞고, 켬은 `the→their` 한 단어를 틀렸습니다. 같은 녹음을 tiny는 끔·켬 모두 `the→their`로 틀렸습니다.
- 그래서 tiny의 "켬이 더 낫다"는 **기준을 3개째에서 겨우 넘은 한 녹음의 효과**이고, 전체 효과로 일반화할 수 없습니다. base에서는 방향이 반대입니다. 두 모델의 합(−3 + 1 = −2개)은 252단어에서 노이즈 수준입니다.
- 오류가 나머지 어디에 있는가: 문장 3(0.167)과 6(0.083)은 **VAD와 무관하게 4개 조건에서 똑같이** 틀립니다. 모델·발음·정답의 문제이지 VAD의 문제가 아닙니다.

**3. 예상과 결과 비교 (검토 문서 15.5에 실행 전에 적어 둔 예상)**

| 예상 | 결과 |
|---|---|
| 빈 인식 0건 | **맞음** (0건) |
| 반복 3회가 같은 값 | **맞음** (모든 조건에서 min = max) |
| 오류 단어 차이 3개 미만 → 구분 불가 | **절반만 맞음**: base는 구분 불가, tiny는 딱 3개라 "켬이 더 낫다"로 기준을 넘음(한 녹음 때문) |
| 기본값은 끔 | **맞음** (모델마다 방향이 갈려 (다) → 끔) |

**4. 결정 (이슈 #34의 "VAD 기본값 결정")**
- **`stt.recognize`(글자 인식)의 기본은 끔으로 유지합니다.** 코드를 바꾸지 않습니다. 이유: 말소리에서 켬이 도움이 된다는 일관된 증거가 없고(tiny +3, base −1), 켬은 인식 시간이 약 0.15초(약 10%) 더 걸리며, 끔이 7강 설정이고 노래에서 켬이 위험했기 때문입니다.
- **`stt.recognize_words`(단어 시간)의 기본(켬)은 이 실험으로 바꾸지 않습니다.** 이 설정은 글자가 아니라 **시간 정확도**를 위한 것이고 실험4는 시간을 재지 않았습니다(검토 문서 15.5의 5). 사전 기준 3(다)의 "두 경로 모두 끔"은 **결과를 읽기 전에 15.5에서 `recognize`에 대한 결정으로 좁혀 읽기로 적었고**, 그에 따라 통일하지 않습니다. **결과에 맞춰 기준을 바꾼 것이 아니라는 점을 밝혀 둡니다**(15.5는 내가 결과를 읽기 전에 썼습니다). 두 경로가 서로 다른 기본값을 갖는 것은 **역할이 다르기 때문**이라고 코드 설명에 적어 두는 것이 이번 결정의 결과입니다.
- **후속(제안)**: `recognize_words`의 VAD 끔/켬에서 **첫 단어 시작 시각**이 달라지는지를 소리의 실제 시작(에너지 기준)과 대조해 확인합니다 **[미확인]**. 이슈 #34 안에서 하거나 별도 이슈로 뺄지는 사용자가 정합니다.

**5. 노래 결과와 나란히 놓으면 (합치지 않고 나열, 검토 문서 15.5의 3)**

| 입력 | VAD 켬의 효과 | 빈 인식 | 출처 |
|---|---|---|---|
| 말소리(내 따라읽기, 반주 없음) | **영향 작음**(tiny +3 / base −1 오류, 각각 녹음 1개) | **0건** | 이 실험 |
| 노래 보컬(반주를 뗀 목소리) | 도움 안 됨(base 0.499 대 0.382, tiny 0.602 대 0.586) | 없음 | 절차 문서 8.2 설정 B |
| 노래 원곡(반주 포함) | **가사를 통째로 지움** | 8구간 중 5구간 | 절차 문서 8.2 설정 B |

"VAD는 말소리용이라 음악이 섞이면 가사를 지운다"는 가설은 **부분적으로 지지**됩니다: 말소리에서 지우지 않았다는 점은 맞지만, 말소리에서 **도움이 된다**는 증거는 없습니다("해가 없다"까지만). 정확도를 높이려면 음성만 넘기라는 면담의 조언은 이 데이터에서는 **말소리에서도 효과를 확인하지 못했고**, 노래에서는 오히려 위험했습니다.

**6. 한계와 이 결과를 쓸 때 주의할 점**
- **바닥 효과**: 말소리의 오류가 매우 적어(252단어 중 6~11개) VAD의 작은 효과를 **잴 힘이 작습니다** **[추정]**. "끔이 더 낫다"가 아니라 **"이 데이터로는 구분할 수 없다"** 가 정확한 읽기입니다.
- 한 사람, 마이크·환경 하나, 문장 10개, tiny·base만. `small`·`medium`과 VAD의 관계는 보지 않았습니다(문장 1~3에서 small + VAD 켬이 한 문장만 좋아졌던 값은 검토 문서 15.2). 소음이 있는 환경의 녹음에서는 VAD가 다를 수 있습니다 **[미확인]**.
- **반복은 필요 없었습니다**: 3회가 모든 조건에서 같았으므로 말소리 인식은 사실상 결정적입니다(온도 폴백이 거의 일어나지 않음 **[추정]**). 이 셀의 `REPEATS`를 1로 줄이면 시간이 약 3분의 1로 줄어듭니다(지금 3회 합계 623초, 첫 반복 272초는 모델 읽기·준비가 섞였을 수 있음 **[추정]**). `scripts/check_notebook.py`로 노트북 전체를 점검할 때도 이 셀이 오래 걸리므로 줄이는 것을 권합니다.
- 이 셀의 출력에는 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.

**7. 이슈 #34 완료 기준 점검**
- [x] 문장 × 회차 WER 표와 Micro/Macro 요약, 인식 시간 (위 [1]·[3])
- [x] 차이가 반복 범위보다 큰지 판정 (tiny는 기준을 겨우 넘음 = 녹음 1개, base는 구분 불가)
- [x] 말소리에서 VAD가 지우는 건수 (**0건**)
- [x] VAD 기본값 결정 (`recognize` 끔 유지, `recognize_words`는 시간 확인 뒤, 코드 변경 없음)
- [ ] 결과와 해석을 검토 문서 15.2에 기록 (다음에 할 일)

## S4. 연습 흐름: 듣고 → 따라읽기 또는 받아쓰기 → 채점

한 문장을 위에서 아래로 한 번에 연습하는 화면입니다. 지금까지 만든 부분(TTS 캐시, 녹음 점검, Whisper 인식, `scoring`)을 `practice()` 하나로 이어 붙였습니다. 이 셀은 보여 주는 일만 하고, **순서는 `practice.py`, 글은 `report.py`** 가 맡습니다(같은 함수를 나중에 웹 화면에서도 부를 수 있도록 모듈로 분리).

| 모드 | 순서 |
|---|---|
| `"read"` 따라읽기 | 문장과 뜻 확인 → 기준 음성 듣기 → **녹음 상태 점검** → (미리 녹음한 파일을) 인식 → 오류율과 틀린 부분 |
| `"dictation"` 받아쓰기 | 기준 음성 듣기(문장은 숨김) → 들은 대로 입력 → 오류율과 틀린 부분 → 정답과 뜻 공개 |

- 문장은 `sentences.csv`의 번호(`1`, `2`, `3`)로 부르거나, 직접 쓴 영어 문장을 문자열로 넣을 수 있습니다.
- **직접 입력한 문장은 TTS API로 전송되고 과금됩니다**(한 번 만든 음성은 `data/tts_cache/`에 저장해 다시 만들지 않고 재사용합니다). 노래 가사를 넣어도 되지만(11-04 규칙 변경) 가사와 만든 음성은 `data/`에만 두고 커밋하지 않습니다.
- 점수는 발음 점수가 아니라 문장 인식 일치도의 오류율입니다. 받아쓰기는 STT를 거치지 않아 인식 오류가 섞이지 않고, 맞춤법·띄어쓰기 같은 내 쓰기 실수만 점수에 반영됩니다.
- 녹음 상태 점검의 경고는 안내일 뿐이며, 경고가 있어도 인식과 채점은 그대로 진행합니다.
- 받아쓰기의 `input()` 창은 VS Code에서 화면 위쪽에 나타납니다.

In [ ]:
# ── S4-0. 연습 한 번의 화면: run_practice 정의 ─────────────────────────────
# 역할       : run_practice(문장, 모드)를 정의합니다. 한 문장 연습 전체(듣기 → 녹음 점검 → 인식 → 채점 → 글 출력)를 이 함수 하나로 부릅니다.
# 원칙       : 이 함수는 화면에 보여 주는 일(print·재생 버튼·input)만 합니다. 순서는 practice.py(flow), 글은 report.py가 맡습니다.
# 호출 경로    : read      : flow.prepare → report.header → TTS 재생 → flow.read(녹음 점검 → stt.recognize → scoring.evaluate) → report.result
#            dictation : flow.prepare → report.header → TTS 재생 → input() → flow.dictation(scoring.evaluate) → report.result
# STT 모델   : read 모드는 stt.recognize (tiny, VAD 끔) / dictation 모드는 STT를 쓰지 않음
# 과금       : 기준 음성이 캐시에 없는 문장(직접 입력한 새 문장)일 때만 TTS 1회
# 가져오는 것   : flow·report (틀린 구간 표시 도우미 셀), sentences·Audio·display (S1-1)
def run_practice(target, mode="read", recording=None):
    """한 문장 연습. target: sentences.csv의 id 또는 직접 쓴 문장.
    mode="read": 문장 확인 → 듣기 → 녹음 점검 → 인식 → 채점 (recording을 안 주면 data/recordings/sNN_take1.*)
    mode="dictation": 듣기(문장은 숨김) → 들은 대로 입력 → 채점 → 정답 공개
    이 셀은 화면에 보여 주는 일만 한다. 순서는 practice.py(flow), 글은 report.py가 만든다."""
    if mode not in ("read", "dictation"):
        raise ValueError('mode는 "read" 또는 "dictation"입니다.')
    # 문장 찾기 + 기준 음성 준비. 직접 쓴 문장이면 TTS API로 나갑니다(캐시에 없을 때)
    prep = flow.prepare(target, sentences)            # 문장 찾기 + 기준 음성(TTS, 캐시)
    if prep["error"]:
        print(prep["error"])
        return
    # report.py: 제목·문장·뜻 글 만들기 (받아쓰기는 문장을 숨김)
    print(report.header(prep, mode))
    print(report.tts_status(prep))
    # 기준 음성 재생 버튼
    display(Audio(filename=str(prep["tts_path"])))

    if mode == "read":
        # 따라읽기: 녹음 상태 점검 → STT 인식 → 채점. 결과는 dict {quality, hyp, seconds, score, ...}
        res = flow.read(prep, recording)               # 녹음 점검 → 인식 → 채점
    else:
        # 받아쓰기: 내가 입력한 글을 바로 채점(STT 없음)
        res = flow.dictation(prep, input("들은 대로 적어 보세요 > "))
    # report.py: 점검·정답·인식·오류율·틀린 부분을 한 덩어리 글로
    print(report.result(prep, res))

In [ ]:
# ── S4-1. 따라읽기 실행: 3번 문장 ─────────────────────────────────────────
# 역할       : 3번 문장을 듣고, data/recordings/s03_take1.wav를 인식·채점합니다(TTS는 캐시, API 0회).
# STT 모델   : tiny (flow.read → stt.recognize), VAD 끔
# 1) 따라읽기: 3번 문장을 듣고, 미리 녹음한 data/recordings/s03_take1.wav를 인식해 채점합니다 (TTS는 캐시, 호출 0회).
run_practice(3, "read")

In [ ]:
# ── S4-2. 받아쓰기 실행: 1번 문장 ──────────────────────────────────────────
# 역할       : 기준 음성을 듣고 들은 대로 입력해 정답과 비교합니다. 입력창이 화면 위쪽에 뜹니다.
# STT 모델   : 사용 안 함 (내가 적은 글을 바로 채점) / TTS는 캐시, API 0회
# 2) 받아쓰기: 1번 문장을 듣고 들은 대로 입력합니다 (입력창은 화면 위쪽에 뜹니다. TTS는 캐시, 호출 0회).
run_practice(1, "dictation")

In [ ]:
# ── S4-3. 직접 쓴 문장으로 받아쓰기 ──────────────────────────────────────────
# 역할       : sentences.csv에 없는 문장도 연습할 수 있습니다. 직접 쓴 영어 문장이나 노래 가사(수정본)를 넣을 수 있습니다.
# 과금       : 처음 쓰는 문장은 TTS 음성을 새로 만들어 API 1회 과금됩니다(한 번 만들면 캐시).
# STT 모델   : 사용 안 함
# 3) 직접 쓴 문장으로 받아쓰기: 문장을 바꿔 쓸 수 있습니다. 처음 쓰는 문장은 TTS를 새로 만들어 과금(1회)됩니다.
#    입력한 문장은 TTS API로 전송됩니다. 노래 가사의 텍스트·음성은 data/에만 두고 커밋하지 않습니다.
run_practice("Where the river meets the sea.", "dictation")

# p.s. 기타 실험

In [ ]:
# ── 녹음 상태 점검 테스트 1: 파일 하나 ─────────────────────────────────────────
# 역할       : s01_take1.wav 하나의 상태(길이·음량·클리핑·무음)를 점검해 봅니다.
# 호출 경로    : quality.check_file(경로) → audio.load_raw로 원본 읽기 → quality.analyze → quality.format_report(글)
# STT 모델   : 사용 안 함 (소리 신호 계산만)
# 2026.11.1(일) 21h30 __ 테스트
import quality
print(quality.format_report(quality.check_file("data/recordings/s01_take1.wav")))

In [ ]:
# ── 녹음 상태 점검 테스트 2: 기준값 확정용 녹음 묶음 ─────────────────────────────────
# 역할       : data/quality_test/의 연습용 녹음(정상·작게·크게·잘림·무음 등)에 상태 점검을 돌려 경고가 기대대로 나오는지 표로 봅니다.
# 담당 모듈    : quality.py — 경고 기준값(너무 작음·클리핑·대부분 무음·너무 짧음)을 이 표로 확정했습니다(검토 문서 14.3).
# STT 모델   : 사용 안 함
import quality, glob
print(f"{'파일':14s}{'길이':>6s}{'RMS':>8s}{'최대':>6s}{'클리핑':>7s}{'무음':>6s}  경고")
for p in sorted(glob.glob("data/quality_test/*.wav")):
    r = quality.check_file(p); m = r["metrics"]
    print(f"{p.split('/')[-1]:14s}{m['seconds']:5.1f}s{m['rms_dbfs']:7.1f}{m['peak']:6.2f}{m['clip_ratio']*100:6.2f}%{m['silent_ratio']*100:5.0f}%  "
        + (" / ".join(w[:9] for w in r["warnings"]) or "없음"))